# Financial Data Agent Workshop — Full Source (Fraud & AML)

**The complete, DDL-included reference notebook.** Every block the 90-minute path compresses — bootstrap, the `FINANCE` seed, the ONNX models, duality views, Oracle Text, identity policies, the DBFS scratchpad, Oracle MLE, and tool-output offload — is expanded here verbatim.

> ⏱️ **This is not the 90-minute path.** Work through [`notebook_student.ipynb`](notebook_student.ipynb) first (five TODOs: `_scan_tables`, `retrieve_knowledge`, `hybrid_rrf_search_memories`, `tool_run_sql`, `agent_turn`). Open this notebook when you want to see *how* the Codespace provisions Oracle, or to deploy this harness against an Oracle that isn't the workshop Codespace.

| Reference | What it is |
|---|---|
| [`notebook_student.ipynb`](notebook_student.ipynb) | The 90-minute, five-TODO learning path |
| [`notebook_complete.ipynb`](notebook_complete.ipynb) | The same five TODOs, already solved |
| [`enterprise_data_agent.ipynb`](enterprise_data_agent.ipynb) | The original end-to-end source notebook |
| [`app/`](app/) | The Flask + React AML app, and `app/scripts/*` — the headless port of this notebook |


## What is in this notebook

This is the *full source* variant. Unlike the 90-minute path it keeps the pre-built DDL and provisioning cells and every advanced chapter:

| Part | Chapter | In the 90-minute path? |
|---|---|---|
| 1 | Setup, connectivity, and Oracle bootstrap (users, vector pool, ONNX models) | condensed into `app/scripts/bootstrap.py` |
| 2 | Long-term memory (OAMP) + the catalog scanner | Block 2 — TODO 1 |
| 3 | Retrieval (vector + rerank + hybrid RRF) and the `FINANCE` seed | Block 3 — TODO 2 / TODO 3 |
| 4 | Toolbox + skillbox, DBFS scratchpad, Oracle MLE | Block 4 — TODO 4 (DBFS/MLE are reference-only) |
| 5 | The agent loop | Block 5 — TODO 5 |
| 7 | JSON Relational Duality Views | advanced reference |
| 9 | Tool-output offload | advanced reference |

Everything that the Codespace does for you in `app/scripts/bootstrap.py`, `seed.py`, and `setup_advanced.py` appears here as executable cells.


# Part 1 — Setup & Connectivity

> 📖 **Guide:** [`docs/part-1-setup.md`](docs/part-1-setup.md)

The setup cells below are **pre-built**. They boot Oracle AI Database (if needed), create the `AGENT` user, configure `vector_memory_size` and `pga_aggregate_limit`, load the ONNX embedder and reranker into the database, and wire up the chat LLM client.

Run them once. They are idempotent — re-running is safe.

> ⚠️ **First run only:** if you see `ACTION REQUIRED: bounce the database` in the output, run `docker restart oracle-free` in a terminal, wait ~60s, and restart this kernel. Subsequent runs detect that `vector_memory_size` is configured and skip the bounce step.

In [ ]:
# 1.0 — Imports. Dependencies are pre-installed in the Codespace; uncomment to install locally.
# %pip install --quiet "oracledb>=2.4" "openai>=1.50" "numpy>=1.26" "tqdm>=4.66" "oracleagentmemory>=26.4" "gdown>=5.2"

import os, time, json, getpass, warnings, subprocess, urllib.request, urllib.error
import tempfile, zipfile, pathlib, shutil, re, hashlib, uuid
import numpy as np
import oracledb
from dataclasses import dataclass

warnings.filterwarnings("ignore", category=UserWarning, module="oracleagentmemory")
print("imports OK — oracledb", oracledb.__version__)

In [ ]:
# 1.1 — Credentials. LLM_PROVIDER picks the chat backend; OPENAI_API_KEY is also
# used by the OAMP extraction LLM regardless of which chat provider you pick.

LLM_PROVIDER = os.environ.setdefault("LLM_PROVIDER", "oci").lower()  # 'oci' or 'openai'
assert LLM_PROVIDER in ("openai", "oci"), "LLM_PROVIDER must be 'openai' or 'oci'"

if LLM_PROVIDER == "oci":
    if not os.environ.get("OCI_GENAI_API_KEY"):
        os.environ["OCI_GENAI_API_KEY"] = getpass.getpass("OCI GenAI API key: ")
    _endpoint = os.environ.get(
        "OCI_GENAI_ENDPOINT", "https://inference.generativeai.us-phoenix-1.oci.oraclecloud.com"
    ).rstrip("/")
    if not (_endpoint.endswith("/openai/v1") or _endpoint.endswith("/20231130/openai")):
        _endpoint += "/v1" if _endpoint.endswith("/openai") else "/openai/v1"
    os.environ["OCI_GENAI_ENDPOINT"] = _endpoint
    os.environ.setdefault("LLM_MODEL", "xai.grok-4.3")
    print(f"Provider: OCI GenAI ({os.environ['LLM_MODEL']})")
else:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
    os.environ.setdefault("LLM_MODEL", "gpt-5.5")
    print(f"Provider: OpenAI ({os.environ['LLM_MODEL']})")

In [ ]:
# 1.2 — Connection helper. Thin retry wrapper around oracledb.connect — used everywhere.

SYS_DSN    = "localhost:1521/FREEPDB1"
SYS_USER   = "sys"
SYS_PASS   = "OraclePwd_2025"
AGENT_USER = "AGENT"
AGENT_PASS = "AgentPwd_2025"


def connect(user: str, password: str, dsn: str, mode: int | None = None,
            retries: int = 5) -> oracledb.Connection:
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            kwargs = dict(user=user, password=password, dsn=dsn)
            if mode is not None:
                kwargs["mode"] = mode
            conn = oracledb.connect(**kwargs)
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE rownum = 1")
                print(f"connected as {user}@{dsn} — {cur.fetchone()[0]}")
            return conn
        except Exception as e:
            last_err = e
            print(f"  attempt {attempt}/{retries} failed: {e}")
            time.sleep(3)
    raise RuntimeError(f"could not connect to {dsn}: {last_err}")

In [ ]:
# 1.3 — Bootstrap (SYS): create AGENT user, allocate vector memory, raise PGA.
# Idempotent — safe to re-run. May print 'ACTION REQUIRED: bounce' on first run.

sys_conn = connect(SYS_USER, SYS_PASS, SYS_DSN, mode=oracledb.AUTH_MODE_SYSDBA)

bootstrap_stmts = [
    f"DECLARE n NUMBER; BEGIN "
    f"  SELECT COUNT(*) INTO n FROM all_users WHERE username = '{AGENT_USER}'; "
    f"  IF n = 0 THEN EXECUTE IMMEDIATE 'CREATE USER {AGENT_USER} IDENTIFIED BY {AGENT_PASS}'; END IF; "
    f"END;",
    f"GRANT CONNECT, RESOURCE, CREATE SESSION TO {AGENT_USER}",
    f"GRANT CREATE TABLE, CREATE SEQUENCE, CREATE VIEW, CREATE PROCEDURE TO {AGENT_USER}",
    f"GRANT UNLIMITED TABLESPACE TO {AGENT_USER}",
    f"GRANT SELECT ON SYS.V_$SQL TO {AGENT_USER}",
    f"GRANT SELECT_CATALOG_ROLE TO {AGENT_USER}",
    f"GRANT EXECUTE ON DBMS_MLE TO {AGENT_USER}",
    f"GRANT DB_DEVELOPER_ROLE TO {AGENT_USER}",
    f"GRANT EXECUTE DYNAMIC MLE TO {AGENT_USER}",
    f"GRANT CTXAPP TO {AGENT_USER}",
    f"GRANT CREATE MINING MODEL TO {AGENT_USER}",
    f"GRANT SELECT ANY TABLE TO {AGENT_USER}",
]

with sys_conn.cursor() as cur:
    for stmt in bootstrap_stmts:
        try:
            cur.execute(stmt)
        except oracledb.DatabaseError as e:
            if e.args[0].code not in (1031,):
                print("  skip:", str(e).splitlines()[0][:80])
sys_conn.commit()
print("AGENT user provisioned with required grants.")

# Vector memory pool — required for HNSW vector indexes.
TARGET_VECTOR_MEMORY = 512 * 1024 * 1024
with sys_conn.cursor() as cur:
    cur.execute("SELECT value FROM v$parameter WHERE name = 'vector_memory_size'")
    current = int(cur.fetchone()[0] or 0)
if current >= TARGET_VECTOR_MEMORY:
    print(f"vector_memory_size already configured ({current/1024/1024:.0f} MiB) — OK.")
else:
    with sys_conn.cursor() as cur:
        cur.execute("ALTER SYSTEM SET vector_memory_size = 512M SCOPE=SPFILE CONTAINER=ALL")
    sys_conn.commit()
    print("vector_memory_size set to 512M (SPFILE).")
    print("ACTION REQUIRED: docker restart oracle-free, then restart this kernel.")

# PGA limit — DBMS_VECTOR.RERANK needs more than the Free image default.
TARGET_PGA = 4 * 1024 * 1024 * 1024
with sys_conn.cursor() as cur:
    cur.execute("SELECT value FROM v$parameter WHERE name = 'pga_aggregate_limit'")
    pga_now = int(cur.fetchone()[0] or 0)
if pga_now < TARGET_PGA:
    try:
        with sys_conn.cursor() as cur:
            cur.execute("ALTER SYSTEM SET pga_aggregate_limit = 4G SCOPE=BOTH")
        sys_conn.commit()
        print("pga_aggregate_limit raised to 4 GiB.")
    except oracledb.DatabaseError as e:
        print(f"  pga_aggregate_limit raise failed (ORA-{e.args[0].code:05d}); "
              "reranker may fall back to cosine ordering under load.")
else:
    print(f"pga_aggregate_limit already configured ({pga_now/1024/1024/1024:.1f} GiB).")

## Connect as the `AGENT` user

> 📖 **See:** [Part 1 guide → Setup checkpoint 1](docs/part-1-setup.md#todo-1-connect-to-oracle-as-the-agent-user)

The pre-built `connect()` function takes `(user, password, dsn)`. Use it to open `agent_conn` — the connection every later cell uses.

The constants `AGENT_USER`, `AGENT_PASS`, and `SYS_DSN` are already defined.


In [ ]:
# Setup checkpoint 1 open `agent_conn` as the AGENT user.
# Hint: agent_conn = connect(AGENT_USER, AGENT_PASS, SYS_DSN)

agent_conn = connect(AGENT_USER, AGENT_PASS, SYS_DSN)

In [ ]:
# ✅ Checkpoint: Setup checkpoint 1
assert "agent_conn" in dir() and agent_conn is not None, (
    "❌ Setup checkpoint 1 incomplete — agent_conn is not defined. See docs/part-1-setup.md."
)
with agent_conn.cursor() as cur:
    cur.execute("SELECT user FROM dual")
    assert cur.fetchone()[0] == "AGENT", "Connected as wrong user — should be AGENT."
print("✅ Setup checkpoint 1 passed — agent_conn is the AGENT user")


### Pre-built — load ONNX embedder and reranker into Oracle

The cell below downloads two ONNX models, copies them into the `oracle-free` container, and registers them in Oracle's mining-model registry. After it runs:

- `VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :text AS DATA)` — 384-dim sentence embedder, called from SQL.
- `PREDICTION(RERANKER_ONNX USING :q AS DATA1, :doc AS DATA2)` — cross-encoder reranker, called from SQL.

Both models live inside the database — same trust boundary as your data, no network round-trips for embedding.

In [ ]:
# 1.4 — Download + load ONNX embedder. Idempotent.

CONTAINER_NAME       = "oracle-free"
CONTAINER_MODEL_DIR  = "/opt/oracle/onnx_models"
ONNX_FILE            = "all_MiniLM_L12_v2.onnx"
ONNX_DIRECTORY       = "ONNX_DIR"
ONNX_EMBED_MODEL     = "ALL_MINILM_L12_V2"
ORACLE_MODEL_URL = (
    "https://adwc4pm.objectstorage.us-ashburn-1.oci.customer-oci.com"
    "/p/TtH6hL2y25EypZ0-rrczRZ1aXp7v1ONbRBfCiT-BDBN8WLKQ3lgyW6RxCfIFLdA6"
    "/n/adwc4pm/b/OML-ai-models/o/all_MiniLM_L12_v2_augmented.zip"
)
os.environ["PATH"] = ":".join(["/opt/homebrew/bin", "/usr/local/bin", "/usr/bin", "/bin",
                                os.environ.get("PATH", "")])
CONTAINER_CLI = "docker" if shutil.which("docker") else "podman"


def _exists_in_container(path):
    return subprocess.run(
        [CONTAINER_CLI, "exec", CONTAINER_NAME, "test", "-f", path],
        capture_output=True,
    ).returncode == 0


subprocess.run([CONTAINER_CLI, "exec", CONTAINER_NAME, "mkdir", "-p", CONTAINER_MODEL_DIR],
               check=True)

target_path = f"{CONTAINER_MODEL_DIR}/{ONNX_FILE}"
if _exists_in_container(target_path):
    print(f"'{ONNX_FILE}' already in {CONTAINER_NAME} — skipping download.")
else:
    print("Downloading Oracle augmented all-MiniLM-L12-v2 ONNX model (~117 MB)...")
    with tempfile.TemporaryDirectory() as tmp:
        zip_path = os.path.join(tmp, "model.zip")
        urllib.request.urlretrieve(ORACLE_MODEL_URL, zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmp)
        onnx_path = next(pathlib.Path(tmp).glob("*.onnx"))
        subprocess.run([CONTAINER_CLI, "cp", str(onnx_path),
                        f"{CONTAINER_NAME}:{target_path}"], check=True)
        subprocess.run([CONTAINER_CLI, "exec", "--user", "0", CONTAINER_NAME,
                        "chmod", "644", target_path], check=False, capture_output=True)
    print(f"  copied to {CONTAINER_NAME}:{target_path}")

# Map an Oracle directory at the container path so DBMS_VECTOR.LOAD_ONNX_MODEL can read it.
with sys_conn.cursor() as cur:
    cur.execute(f"CREATE OR REPLACE DIRECTORY {ONNX_DIRECTORY} AS '{CONTAINER_MODEL_DIR}'")
    cur.execute(f"GRANT READ, WRITE ON DIRECTORY {ONNX_DIRECTORY} TO {AGENT_USER}")
sys_conn.commit()

# Register the embedder.
with agent_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM user_mining_models WHERE model_name = :m",
                m=ONNX_EMBED_MODEL)
    already_loaded = cur.fetchone()[0] > 0

if already_loaded:
    print(f"model {ONNX_EMBED_MODEL!r} already loaded.")
else:
    print(f"loading ONNX model {ONNX_EMBED_MODEL!r}...")
    with agent_conn.cursor() as cur:
        cur.execute(
            "BEGIN DBMS_VECTOR.LOAD_ONNX_MODEL("
            "  directory => :d, file_name => :f, model_name => :m, "
            "  metadata => JSON('{\"function\":\"embedding\",\"embeddingOutput\":\"embedding\","
            "  \"input\":{\"input\":[\"DATA\"]}}')); END;",
            d=ONNX_DIRECTORY, f=ONNX_FILE, m=ONNX_EMBED_MODEL,
        )
    agent_conn.commit()
    print(f"loaded {ONNX_EMBED_MODEL!r}.")

# Smoke test — produce one embedding and report dimension.
with agent_conn.cursor() as cur:
    cur.execute(f"SELECT VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :t AS DATA) FROM dual",
                t="Oracle in-database embedding round-trip.")
    vec = cur.fetchone()[0]
ONNX_EMBED_DIM = len(vec)
print(f"VECTOR_EMBEDDING produced a {ONNX_EMBED_DIM}-dim vector.")

In [ ]:
# 1.5 — Download + load ONNX reranker (optional but recommended).
# If RERANKER_URL is empty, the rerank() helper falls back to cosine ordering.

DEFAULT_RERANKER_URL = (
    "https://drive.google.com/file/d/1-xDRSHr_ulbO7MqVlWLu6ZA2J-bCjUoY/view?usp=drive_link"
)
RERANKER_MODEL = os.environ.get("RERANKER_MODEL_NAME", "RERANKER_ONNX")
RERANKER_URL   = os.environ.get("RERANKER_URL", DEFAULT_RERANKER_URL).strip()
RERANKER_FILE  = os.environ.get("RERANKER_FILE", "bge_reranker_base.onnx")


def _reranker_loaded() -> bool:
    with agent_conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM user_mining_models WHERE model_name = :m",
                    m=RERANKER_MODEL)
        return cur.fetchone()[0] > 0


if _reranker_loaded():
    print(f"reranker {RERANKER_MODEL!r} already loaded.")
elif not RERANKER_URL:
    print("RERANKER_URL not set — skipping reranker load. rerank() will pass through.")
else:
    target = f"{CONTAINER_MODEL_DIR}/{RERANKER_FILE}"
    if not _exists_in_container(target):
        print(f"downloading reranker from {RERANKER_URL}...")
        with tempfile.TemporaryDirectory() as tmp:
            local_path = os.path.join(tmp, RERANKER_FILE)
            drive_match = re.search(r"drive\.google\.com/.*?/d/([A-Za-z0-9_-]+)", RERANKER_URL)
            if drive_match:
                import gdown
                gdown.download(id=drive_match.group(1), output=local_path, quiet=False)
            else:
                urllib.request.urlretrieve(RERANKER_URL, local_path)
            src_path = local_path
            if local_path.endswith(".zip"):
                with zipfile.ZipFile(local_path) as zf:
                    zf.extractall(tmp)
                src_path = str(next(pathlib.Path(tmp).glob("*.onnx")))
            subprocess.run([CONTAINER_CLI, "cp", src_path, f"{CONTAINER_NAME}:{target}"], check=True)
            subprocess.run([CONTAINER_CLI, "exec", "--user", "0", CONTAINER_NAME,
                            "chmod", "644", target], check=False, capture_output=True)
    rerank_meta = json.dumps({
        "function": "regression", "regressionOutput": "output",
        "input": {"first_input": ["DATA1"], "second_input": ["DATA2"]},
    })
    print(f"loading reranker {RERANKER_MODEL!r}...")
    with agent_conn.cursor() as cur:
        cur.execute(
            "BEGIN DBMS_VECTOR.LOAD_ONNX_MODEL("
            "  directory => :d, file_name => :f, model_name => :m, metadata => JSON(:meta)); END;",
            d=ONNX_DIRECTORY, f=RERANKER_FILE, m=RERANKER_MODEL, meta=rerank_meta,
        )
    agent_conn.commit()
    print(f"loaded {RERANKER_MODEL!r}.")

print(f"reranker available: {_reranker_loaded()}")

In [ ]:
# 1.6 — rerank() helper. Calls Oracle's PREDICTION() against the loaded reranker;
# falls through to cosine ordering when no model is loaded so call sites are uniform.

def rerank(query: str, candidates: list[dict], top_k: int = 5,
           content_key: str = "body") -> list[dict]:
    if not candidates:
        return candidates
    if not _reranker_loaded():
        return candidates[:top_k]

    docs = [{"index": i, "content": str(c.get(content_key, ""))[:1500]}
            for i, c in enumerate(candidates)]

    sql = (
        f"SELECT t.idx, "
        f"       PREDICTION({RERANKER_MODEL} USING :q AS DATA1, t.content AS DATA2) AS score "
        "  FROM JSON_TABLE(:docs, '$[*]' COLUMNS ("
        "         idx     NUMBER          PATH '$.index', "
        "         content VARCHAR2(4000)  PATH '$.content'"
        "       )) t "
        " ORDER BY score DESC FETCH FIRST :k ROWS ONLY"
    )
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql, q=query, docs=json.dumps(docs), k=top_k)
            ranked = list(cur)
    except oracledb.DatabaseError as e:
        print(f"  rerank failed — falling back to cosine order: {e}")
        return candidates[:top_k]

    out = []
    for idx, score in ranked:
        if idx is None or int(idx) >= len(candidates):
            continue
        item = dict(candidates[int(idx)])
        item["rerank_score"] = float(score) if score is not None else 0.0
        out.append(item)
    return out


# Smoke test
_demo = [
    {"body": "Oracle AI Database supports vector search natively."},
    {"body": "Bananas are yellow."},
    {"body": "PREDICTION() rescores documents against a query using a loaded reranker."},
]
print("rerank smoke test:")
for hit in rerank("how does in-database reranking work?", _demo, top_k=3):
    print(f"  {hit.get('rerank_score', 0.0):>10.4f}  {hit['body']}")

In [ ]:
# 1.7 — Chat LLM client. Uses the standard openai SDK pointed at OpenAI directly,
# or at OCI GenAI's OpenAI-compatible endpoint depending on LLM_PROVIDER.

from openai import OpenAI

LLM_MODEL = os.environ["LLM_MODEL"]

if LLM_PROVIDER == "oci":
    llm = OpenAI(
        base_url=os.environ["OCI_GENAI_ENDPOINT"],
        api_key=os.environ["OCI_GENAI_API_KEY"],
    )
else:
    llm = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def chat(messages: list[dict], tools: list[dict] | None = None,
         model: str = LLM_MODEL, max_retries: int = 3):
    """Call the LLM with retry on 429."""
    kwargs = {"model": model, "messages": messages}
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"
    delay = 2.0
    for attempt in range(max_retries + 1):
        try:
            return llm.chat.completions.create(**kwargs)
        except Exception as e:
            status = getattr(e, "status_code", None) or 0
            if status == 429 and attempt < max_retries:
                print(f"  429 — retrying in {delay:.0f}s")
                time.sleep(delay); delay *= 2
            else:
                raise


resp = chat([
    {"role": "system", "content": "Be terse."},
    {"role": "user",   "content": "Say 'pong'."},
])
print(f"[{LLM_PROVIDER}/{LLM_MODEL}]", resp.choices[0].message.content)

# Part 2 — Long-Term Memory with OAMP

> 📖 **Guide:** [`docs/part-2-oamp-memory.md`](docs/part-2-oamp-memory.md)

> 🔧 **TODO in this part:** **TODO 1** — `_scan_tables`

The long-term store **is** the [Oracle AI Agent Memory Package (OAMP)](https://www.oracle.com/database/ai-agent-memory/). Instead of hand-rolling memory tables, we hand a connection to `OracleAgentMemory` and let it own the DDL, the embedding pipeline, and the retrieval surface.

| OAMP primitive | What it stores |
|---|---|
| `memory` | Durable facts — scanned schema entries, corrections, tool outputs |
| `thread` | A conversation. Holds messages, exposes a context card |
| `context_card` | Compact, query-relevant block of memories + recent turns |

We keep one bespoke table — `scan_history` — for **procedural** memory of the agent's own scans (queried by time/owner, not by meaning).

In [ ]:
# 2.1 — In-DB ONNX embedder for OAMP. Every embed() call issues a SELECT VECTOR_EMBEDDING
# against agent_conn — zero network calls, embedding happens in the same process as the database work.

from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.llms import Llm
from oracleagentmemory.apis.embedders.embedder import IEmbedder


class OracleONNXEmbedder(IEmbedder):
    def __init__(self, conn, model_name: str = ONNX_EMBED_MODEL, dim: int = ONNX_EMBED_DIM):
        self._conn = conn
        self._model = model_name
        self._dim = dim

    def embed(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        out = np.zeros((len(texts), self._dim), dtype=np.float32)
        sql = f"SELECT VECTOR_EMBEDDING({self._model} USING :t AS DATA) FROM dual"
        with self._conn.cursor() as cur:
            for i, t in enumerate(texts):
                cur.execute(sql, t=t)
                out[i] = np.asarray(list(cur.fetchone()[0]), dtype=np.float32)
        return out

    async def embed_async(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        return self.embed(texts, is_query=is_query)


# Wire OAMP's extraction LLM to whichever chat provider §1 picked.
if LLM_PROVIDER == "oci":
    extraction_llm = Llm(f"openai/{LLM_MODEL}",
                         api_base=os.environ["OCI_GENAI_ENDPOINT"],
                         api_key=os.environ["OCI_GENAI_API_KEY"])
else:
    extraction_llm = Llm(LLM_MODEL)

memory_client = OracleAgentMemory(
    connection=agent_conn,
    embedder=OracleONNXEmbedder(agent_conn),
    llm=extraction_llm,
    extract_memories=True,
    schema_policy="create_if_necessary",
    table_name_prefix="eda_onnx_",
)
print("OAMP client wired (in-DB ONNX embedder + extraction LLM).")

## Register the operator and agent identities

> 📖 **See:** [Part 2 guide → Setup checkpoint 2](docs/part-2-oamp-memory.md#todo-2-register-the-user-and-agent-identities)

Every memory record OAMP stores carries a `user_id` and an `agent_id`. Register both up front so the rest of the notebook can reference them by stable strings.

Wrap each call in `try/except ValueError` so re-running doesn't blow up on "already exists".


In [ ]:
USER_ID  = "enterprise-operator"
AGENT_ID = "enterprise-data-agent"

# Setup checkpoint 2 Register USER_ID and AGENT_ID with OAMP.
# Use memory_client.add_user(USER_ID, info) and memory_client.add_agent(AGENT_ID, info).
# Wrap each in try/except ValueError so re-runs don't fail on "already exists".

for register_fn, eid, info in [
    (memory_client.add_user,  USER_ID,  "Operator querying the enterprise database in natural language."),
    (memory_client.add_agent, AGENT_ID, "Data agent grounded in scanned schema metadata."),
]:
    try:
        register_fn(eid, info)
        print(f"registered {eid}")
    except ValueError as e:
        if "already exists" in str(e):
            print(f"(already exists) {eid}")
        else:
            raise

In [ ]:
# ✅ Checkpoint: Setup checkpoint 2
_users = memory_client._store.list_users() if hasattr(memory_client._store, "list_users") else None
print("✅ Setup checkpoint 2 passed — user/agent registered with OAMP")


In [ ]:
# 2.2 — scan_history: procedural memory of the agent's own scans.
# Queried by time/owner, not by meaning, so it stays a regular table (not OAMP).

with agent_conn.cursor() as cur:
    try:
        cur.execute(
            "CREATE TABLE scan_history ("
            "  scan_id          VARCHAR2(64) DEFAULT SYS_GUID() PRIMARY KEY,"
            "  target_owner     VARCHAR2(128) NOT NULL,"
            "  objects_scanned  NUMBER,"
            "  facts_written    NUMBER,"
            "  notes            VARCHAR2(4000),"
            "  started_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP,"
            "  finished_at      TIMESTAMP)"
        )
        print("created scan_history")
    except oracledb.DatabaseError as e:
        if e.args[0].code != 955:
            raise
        print("(already exists) scan_history")
agent_conn.commit()

## The Schema Scanner — Catalog Views as Training Data

Four scanners, each mining one catalog source into a `list[Fact]`. The simplest one is `_scan_tables` — your TODO. The other three (`_scan_columns`, `_scan_relationships`, `_scan_workload`) follow the same shape and are pre-built.

In [ ]:
@dataclass
class Fact:
    kind: str        # 'table' | 'column' | 'relationship' | 'query_pattern'
    subject: str     # e.g. 'FINANCE.VESSELS'
    body: str        # natural-language sentence the embedder will read
    metadata: dict   # owner, table, column, etc.

## Implement `_scan_tables`

> 📖 **See:** [Part 2 guide → TODO 1](docs/part-2-oamp-memory.md#todo-1-implement-_scan_tables)

Mine `ALL_TABLES + ALL_TAB_COMMENTS` and emit one `Fact(kind="table")` per table. Build the `body` conditionally — skip parts that are `None`.


In [ ]:
# TODO 1: implement _scan_tables(conn, owner) -> list[Fact]
# Query: SELECT table_name, comments, num_rows, last_analyzed
#   FROM all_tables LEFT JOIN all_tab_comments USING (owner, table_name)
#   WHERE owner = :owner ORDER BY table_name
# For each row, build a body string and append a Fact(kind="table", subject=f"{owner}.{table}", ...).

def _scan_tables(conn, owner: str) -> list[Fact]:
    sql = (
        "SELECT t.table_name, tc.comments, t.num_rows, t.last_analyzed "
        "  FROM all_tables t "
        "  LEFT JOIN all_tab_comments tc "
        "    ON tc.owner = t.owner AND tc.table_name = t.table_name "
        " WHERE t.owner = :owner "
        " ORDER BY t.table_name"
    )
    facts: list[Fact] = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for table, comment, num_rows, last_analyzed in cur:
            body_parts = [f"Table {owner}.{table}."]
            if comment:
                body_parts.append(f"Documented purpose: {comment}")
            if num_rows is not None:
                body_parts.append(f"Approximate row count: {num_rows:,}.")
            if last_analyzed:
                body_parts.append(f"Statistics last gathered at {last_analyzed}.")
            facts.append(Fact(
                kind="table",
                subject=f"{owner}.{table}",
                body=" ".join(body_parts),
                metadata={"owner": owner, "table": table,
                          "num_rows": num_rows, "has_comment": bool(comment)},
            ))
    return facts

In [ ]:
# ✅ Checkpoint: TODO 1
assert callable(_scan_tables), "❌ TODO 1 incomplete — _scan_tables not defined"
# We can't fully test this until FINANCE exists (next cell), but at least the function shape is right.
import inspect as _ins
_sig = _ins.signature(_scan_tables)
assert list(_sig.parameters) == ["conn", "owner"], "❌ Wrong signature for _scan_tables"
print("✅ TODO 1 passed — _scan_tables defined with correct signature")

### Pre-built — the other three scanners

`_scan_columns`, `_scan_relationships`, `_scan_workload` follow the same shape as your `_scan_tables`. Read them once and notice how each catalog view becomes one English sentence the embedder can index.

In [ ]:
def _scan_columns(conn, owner: str) -> list[Fact]:
    sql = (
        "SELECT c.table_name, c.column_name, c.data_type, c.data_length, "
        "       c.nullable, cc.comments "
        "  FROM all_tab_columns c "
        "  LEFT JOIN all_col_comments cc "
        "    ON cc.owner = c.owner AND cc.table_name = c.table_name "
        "   AND cc.column_name = c.column_name "
        " WHERE c.owner = :owner "
        " ORDER BY c.table_name, c.column_id"
    )
    facts = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for table, col, dtype, dlen, nullable, comment in cur:
            dtype_str = dtype + (f"({dlen})" if dlen and dtype in ("VARCHAR2", "CHAR") else "")
            nullstr = "nullable" if nullable == "Y" else "NOT NULL"
            body = f"Column {owner}.{table}.{col} of type {dtype_str} ({nullstr})."
            if comment:
                body += f" Meaning: {comment}"
            facts.append(Fact(
                kind="column",
                subject=f"{owner}.{table}.{col}",
                body=body,
                metadata={"owner": owner, "table": table, "column": col,
                          "data_type": dtype, "nullable": nullable == "Y"},
            ))
    return facts


def _scan_relationships(conn, owner: str) -> list[Fact]:
    sql = (
        "SELECT c.constraint_name, c.table_name, acc.column_name, "
        "       rc.table_name AS r_table, rcc.column_name AS r_column "
        "  FROM all_constraints c "
        "  JOIN all_cons_columns acc "
        "    ON acc.owner = c.owner AND acc.constraint_name = c.constraint_name "
        "  JOIN all_constraints rc "
        "    ON rc.owner = c.r_owner AND rc.constraint_name = c.r_constraint_name "
        "  JOIN all_cons_columns rcc "
        "    ON rcc.owner = rc.owner AND rcc.constraint_name = rc.constraint_name "
        "   AND rcc.position = acc.position "
        " WHERE c.owner = :owner AND c.constraint_type = 'R'"
    )
    facts = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for cname, tbl, col, r_tbl, r_col in cur:
            facts.append(Fact(
                kind="relationship",
                subject=f"{owner}.{tbl}.{col}->{owner}.{r_tbl}.{r_col}",
                body=(f"Foreign key {cname}: {owner}.{tbl}.{col} references "
                      f"{owner}.{r_tbl}.{r_col}. Use this edge when joining the two tables."),
                metadata={"owner": owner, "child": tbl, "child_col": col,
                          "parent": r_tbl, "parent_col": r_col},
            ))
    return facts


def _scan_workload(conn, owner: str, limit: int = 50) -> list[Fact]:
    sql = (
        "SELECT /*+ FIRST_ROWS(:lim) */ "
        "       sql_id, sql_fulltext, executions, rows_processed "
        "  FROM v$sql "
        " WHERE parsing_schema_name = :owner "
        "   AND command_type IN (3, 6, 7, 189) "
        "   AND rownum <= :lim "
        " ORDER BY executions DESC NULLS LAST"
    )
    facts = []
    with conn.cursor() as cur:
        try:
            cur.execute(sql, owner=owner.upper(), lim=limit)
            for sql_id, text, execs, rows_proc in cur:
                if text is None:
                    continue
                stmt = (text.read() if hasattr(text, "read") else str(text)).strip()
                if len(stmt) > 2000:
                    stmt = stmt[:2000] + " /* ...truncated */"
                facts.append(Fact(
                    kind="query_pattern",
                    subject=f"v$sql:{sql_id}",
                    body=(f"A SQL statement observed in the workload for {owner} "
                          f"(executed {execs or 0} times, {rows_proc or 0} rows processed):\n{stmt}"),
                    metadata={"owner": owner, "sql_id": sql_id,
                              "executions": execs, "rows_processed": rows_proc},
                ))
        except oracledb.DatabaseError as e:
            print(f"  (workload scan skipped — V$SQL access: {e})")
    return facts


def scan_schema(conn, owner: str) -> list[Fact]:
    facts = []
    facts += _scan_tables(conn, owner)
    facts += _scan_columns(conn, owner)
    facts += _scan_relationships(conn, owner)
    facts += _scan_workload(conn, owner)
    return facts


print(f"scanners ready: _scan_tables (your TODO 1), _scan_columns, _scan_relationships, _scan_workload, scan_schema")

In [ ]:
# 2.3 — write_facts: idempotent upsert keyed on (kind, subject) with body_hash dedup.
# run_scan: scan a schema, persist facts, record bookkeeping in scan_history.

def _hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:32]


def write_facts(facts: list[Fact], scan_id: str | None = None) -> tuple[int, int, int]:
    new = updated = skipped = 0
    scan_id = scan_id or str(uuid.uuid4())
    store = memory_client._store

    for f in facts:
        body_hash = _hash(f.body)
        existing = store.list(
            "memory",
            user_id=USER_ID, agent_id=AGENT_ID,
            metadata_filter={"kind": f.kind, "subject": f.subject},
            limit=1,
        )
        meta = {**f.metadata, "kind": f.kind, "subject": f.subject,
                "body_hash": body_hash, "scan_id": scan_id}

        if not existing:
            memory_client.add_memory(f.body, user_id=USER_ID, agent_id=AGENT_ID, metadata=meta)
            new += 1
        elif (existing[0].metadata or {}).get("body_hash") == body_hash:
            skipped += 1
        else:
            memory_client.delete_memory(existing[0].id)
            memory_client.add_memory(f.body, user_id=USER_ID, agent_id=AGENT_ID, metadata=meta)
            updated += 1
    return new, updated, skipped


def run_scan(conn, owner: str) -> dict:
    scan_id = str(uuid.uuid4())
    with conn.cursor() as cur:
        cur.execute("INSERT INTO scan_history (scan_id, target_owner, notes) VALUES (:id, :o, :n)",
                    id=scan_id, o=owner.upper(), n="started")
    conn.commit()
    facts = scan_schema(conn, owner)
    new, updated, skipped = write_facts(facts, scan_id=scan_id)
    with conn.cursor() as cur:
        cur.execute(
            "UPDATE scan_history SET objects_scanned = :n, facts_written = :w, "
            "  finished_at = CURRENT_TIMESTAMP, notes = :notes WHERE scan_id = :id",
            n=len(facts), w=new + updated,
            notes=f"new={new} updated={updated} skipped={skipped}",
            id=scan_id)
    conn.commit()
    return {"scan_id": scan_id, "facts_total": len(facts),
            "new": new, "updated": updated, "skipped": skipped}


print("write_facts + run_scan ready")

# Part 3 — Retrieval Strategies

> 📖 **Guide:** [`docs/part-3-retrieval.md`](docs/part-3-retrieval.md)

> 🔧 **TODOs in this part (2):** **TODO 2** — `retrieve_knowledge`; **TODO 3** — `hybrid_rrf_search_memories`

This Part adds two retrieval surfaces over the OAMP store from Part 2:

1. **Vector search + cross-encoder rerank** — strong on meaning.
2. **Hybrid (vector + Oracle Text) fused via Reciprocal Rank Fusion** — strong on meaning *and* exact tokens.

Before we build them, the next cell seeds a realistic `FINANCE` schema (branches, customers, accounts, cards, merchants, transactions, loans, SAR reports) for the agent to reason over.

### Pre-built — seed the `FINANCE` demo schema

The cell below creates a `FINANCE` user, eight tables (branches, customers, accounts, cards, merchants, transactions, loans, sar_reports), spatial metadata + indexes on the geometry columns, and ~2,100 rows of realistic data. Run it once.

Two columns to remember — these are the exact kind of facts a senior engineer remembers and an LLM hallucinates:

- **`transactions.amount_cents`** is in USD cents, not dollars.
- **`customers.risk_rating`** is a 1-100 score, higher means riskier.

The scanner picks both up via `COMMENT ON COLUMN`.

In [ ]:
# Pivot: replace the toy DEMO schema with the Meridian Bank dataset
# (branches, customers, accounts, cards, merchants, transactions, loans,
# sar_reports). Spatial columns (SDO_GEOMETRY) on `branches.location` and
# `merchants.location`, with R-tree spatial indexes for SDO_WITHIN_DISTANCE
# queries. Same `DEMO_USER` Python variable so downstream cells don't have to
# move; the actual schema in Oracle is `FINANCE`.

DEMO_USER = "FINANCE"
DEMO_PASS = "FinancePwd_2025"

with sys_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM all_users WHERE username = :u", u=DEMO_USER)
    if cur.fetchone()[0] == 0:
        cur.execute(f"CREATE USER {DEMO_USER} IDENTIFIED BY {DEMO_PASS}")
    cur.execute(f"GRANT CONNECT, RESOURCE, UNLIMITED TABLESPACE TO {DEMO_USER}")
    cur.execute(f"GRANT CREATE VIEW, CREATE PROCEDURE, CREATE TYPE TO {DEMO_USER}")
    # Spatial: every user that creates SDO_GEOMETRY columns needs this grant.
    try:
        cur.execute(f"GRANT EXECUTE ON MDSYS.SDO_GEOMETRY TO {DEMO_USER}")
    except oracledb.DatabaseError:
        pass
    cur.execute(f"GRANT SELECT ANY TABLE TO {AGENT_USER}")
sys_conn.commit()

demo_conn = connect(DEMO_USER, DEMO_PASS, SYS_DSN)

# ---- Schema + spatial + seed + duality views, driven by the same module the
# ---- Codespace provisioning runs (app/backend/db/seed_finance.py). Re-running
# ---- this cell drops and re-creates everything.
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(_os.path.join(_os.getcwd(), "..", "app", "backend")))
from db.seed_finance import seed as _seed_finance
_seed_finance(demo_conn)
print(f"FINANCE seeded: {DEMO_USER}")


In [ ]:
# 3.1 — Run the scanner against FINANCE. After this cell, the OAMP store contains
# one memory per table, column, foreign-key, and observed query — all embedded and retrievable.

DEMO_USER = "FINANCE"
summary = run_scan(agent_conn, owner=DEMO_USER)
print(json.dumps(summary, indent=2))

In [ ]:
# 3.2 — Oracle Text index on the OAMP memory body. Required for hybrid retrieval (§3.4).
# SYNC (ON COMMIT) keeps the index fresh as new memories land.

TEXT_INDEX_NAME = "eda_memory_text_idx"
MEMORY_TABLE    = "eda_onnx_memory"

with agent_conn.cursor() as cur:
    try:
        cur.execute(f"DROP INDEX {TEXT_INDEX_NAME}")
        print(f"dropped existing {TEXT_INDEX_NAME}")
    except oracledb.DatabaseError as e:
        if e.args[0].code != 1418:
            raise
    try:
        cur.execute(
            f"CREATE INDEX {TEXT_INDEX_NAME} ON {MEMORY_TABLE}(content) "
            f"  INDEXTYPE IS CTXSYS.CONTEXT PARAMETERS ('SYNC (ON COMMIT)')"
        )
        print(f"created Oracle Text index {TEXT_INDEX_NAME}")
    except oracledb.DatabaseError as e:
        if e.args[0].code == 29855:
            cur.execute(f"CREATE INDEX {TEXT_INDEX_NAME} ON {MEMORY_TABLE}(content) "
                        f"  INDEXTYPE IS CTXSYS.CONTEXT")
            print(f"created {TEXT_INDEX_NAME} (without SYNC ON COMMIT)")
        else:
            raise
agent_conn.commit()

## Implement `retrieve_knowledge`

> 📖 **See:** [Part 3 guide → TODO 2](docs/part-3-retrieval.md#todo-2-implement-retrieve_knowledge)

`retrieve_knowledge(query, k, kinds=None)` is a two-stage call:

1. Cosine search via `memory_client.search` — oversample by 4× so the reranker has enough candidates.
2. Filter out `tool_output` memories so the log of past tool calls doesn't pollute knowledge retrieval.
3. Apply `kinds` filter if provided.
4. Rerank via `rerank(query, candidates, top_k=k, content_key="body")`.


In [ ]:
# TODO 2: implement retrieve_knowledge(query, k=5, kinds=None) -> list[dict]

def retrieve_knowledge(query: str, k: int = 5,
                       kinds: list[str] | None = None) -> list[dict]:
    """Semantic search over the agent's long-term memory."""
    # Defensive: LLMs sometimes pass `kinds` as a comma-separated string.
    if isinstance(kinds, str):
        kinds = [s.strip() for s in kinds.split(",") if s.strip()]
    if kinds == []:
        kinds = None

    cosine_fetch = k * 4
    hits = memory_client.search(
        query,
        user_id=USER_ID, agent_id=AGENT_ID,
        record_types=["memory"],
        max_results=cosine_fetch,
    )

    candidates: list[dict] = []
    for h in hits:
        meta = h.metadata or {}
        kind_value = meta.get("kind")
        if kind_value == "tool_output":
            continue
        if kinds is not None and (kind_value is None or kind_value not in kinds):
            continue
        candidates.append({
            "kind":     kind_value or "?",
            "subject":  meta.get("subject", ""),
            "body":     h.content,
            "metadata": meta,
            "distance": float(h.distance),
        })

    return rerank(query, candidates, top_k=k, content_key="body")

In [ ]:
# ✅ Checkpoint: TODO 2 — quick probe
hits = retrieve_knowledge("which table holds transaction amounts", k=3)
assert len(hits) > 0, "❌ TODO 2 returned no hits — check the scan ran and the function is correct."
for h in hits:
    print(f"  [{h['kind']:12s}] {h['subject'][:40]:40s}  {h['body'][:100]}")
print("\n✅ TODO 2 passed — retrieve_knowledge returns hits")

In [ ]:
# 3.3 — keyword_search_memories: full-text via Oracle Text. Pre-built.

def keyword_search_memories(query: str, k: int = 5) -> list[dict]:
    sql = f"""
        SELECT m.record_id, m.metadata,
               DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content,
               SCORE(1) AS score_txt
          FROM {MEMORY_TABLE} m
         WHERE CONTAINS(m.content, :kw, 1) > 0
           AND m.user_id  = :u AND m.agent_id = :a
           AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
         ORDER BY SCORE(1) DESC
         FETCH FIRST :k ROWS ONLY
    """
    with agent_conn.cursor() as cur:
        cur.execute(sql, kw=query, u=USER_ID, a=AGENT_ID, k=k)
        rows = []
        for record_id, metadata, content, score in cur:
            meta = metadata or {}
            if hasattr(content, "read"):
                content = content.read()
            rows.append({
                "record_id": record_id,
                "kind": meta.get("kind", "memory"),
                "subject": meta.get("subject", ""),
                "content": str(content or "")[:500],
                "score_txt": float(score) if score is not None else 0.0,
            })
    return rows


def hybrid_rrf_search_memories(query: str, k: int = 5,
                                per_list: int = 30, rrf_k: int = 60) -> list[dict]:
    """Hybrid retrieval over OAMP memories: vector + Oracle Text fused via RRF.
    Issued as one SQL statement — JOIN happens server-side."""
    sql = f"""
        WITH q_emb AS (
            SELECT TO_VECTOR(VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), 384, FLOAT64) AS emb
              FROM dual
        ),
        vec AS (
            SELECT m.record_id,
                   DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content,
                   m.metadata,
                   1 - VECTOR_DISTANCE(c.embedding, q_emb.emb, COSINE) AS sim_vec,
                   ROW_NUMBER() OVER (ORDER BY VECTOR_DISTANCE(c.embedding, q_emb.emb, COSINE)) AS r_vec
              FROM {MEMORY_TABLE} m
              JOIN eda_onnx_record_chunks c ON c.source_id = m.record_id
              CROSS JOIN q_emb
             WHERE m.user_id = :u AND m.agent_id = :a
               AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                    OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
             FETCH FIRST :n ROWS ONLY
        ),
        txt AS (
            SELECT m.record_id,
                   DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content,
                   m.metadata,
                   SCORE(1) AS score_txt,
                   ROW_NUMBER() OVER (ORDER BY SCORE(1) DESC) AS r_txt
              FROM {MEMORY_TABLE} m
             WHERE CONTAINS(m.content, :kw, 1) > 0
               AND m.user_id = :u AND m.agent_id = :a
               AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                    OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
             FETCH FIRST :n ROWS ONLY
        ),
        fused AS (
            SELECT COALESCE(v.record_id, t.record_id) AS record_id,
                   COALESCE(v.content, t.content)     AS content,
                   COALESCE(v.metadata, t.metadata)   AS metadata,
                   NVL(v.sim_vec, 0)                  AS sim_vec,
                   NVL(t.score_txt, 0)                AS score_txt,
                   NVL(v.r_vec, 999999)               AS r_vec,
                   NVL(t.r_txt, 999999)               AS r_txt,
                   ( 1.0 / (:rrf_k + NVL(v.r_vec, 999999))
                   + 1.0 / (:rrf_k + NVL(t.r_txt, 999999)) ) AS rrf_score
              FROM vec v
              FULL OUTER JOIN txt t ON v.record_id = t.record_id
        )
        SELECT * FROM fused
         ORDER BY rrf_score DESC
         FETCH FIRST :k ROWS ONLY
    """
    with agent_conn.cursor() as cur:
        kw = f'"{query}"' if " " in query.strip() else query
        cur.execute(sql, q=query, kw=kw, u=USER_ID, a=AGENT_ID,
                    n=per_list, rrf_k=rrf_k, k=k)
        rows = []
        for rec_id, content, meta, sim_vec, score_txt, r_vec, r_txt, rrf in cur:
            if hasattr(content, "read"):
                content = content.read()
            rows.append({
                "record_id": rec_id,
                "kind": (meta or {}).get("kind", "memory"),
                "subject": (meta or {}).get("subject", ""),
                "content": str(content or "")[:500],
                "sim_vec":   float(sim_vec)   if sim_vec   is not None else 0.0,
                "score_txt": float(score_txt) if score_txt is not None else 0.0,
                "r_vec":     int(r_vec),
                "r_txt":     int(r_txt),
                "rrf_score": float(rrf),
            })
    return rows


print("keyword_search_memories + hybrid_rrf_search_memories ready")

## Run the three-way retrieval probe

> 📖 **See:** [Part 3 guide → Demo: three-way retrieval probe](docs/part-3-retrieval.md#todo-5-run-the-three-way-retrieval-probe)

Same query through three retrievers — vector only, keyword only, hybrid via RRF. Watch the `r_vec` / `r_txt` columns in the hybrid output: rows that show up in **both** lists get the highest combined score.


In [ ]:
# Demo: three-way retrieval probe: run three retrievals on the same query and print results.

probe_q = "amount_cents transaction amounts stored in USD CENTS, never dollars"

print("=" * 80)
print(f"QUERY: {probe_q!r}")
print("=" * 80)

print("\n--- A) VECTOR ONLY (retrieve_knowledge) ---")
for h in retrieve_knowledge(probe_q, k=3):
    subject = (h.get("metadata") or {}).get("subject", "")
    print(f"  [{h.get('kind','memory'):12s}] {subject[:40]:40s}  {h['body'][:100]}")

print("\n--- B) KEYWORD ONLY (Oracle Text CONTAINS + SCORE) ---")
for h in keyword_search_memories(probe_q, k=3):
    print(f"  [{h['kind']:12s}] score={h['score_txt']:6.2f}  {h['subject'][:40]:40s}  {h['content'][:100]}")

print("\n--- C) HYBRID via RRF ---")
for h in hybrid_rrf_search_memories(probe_q, k=3):
    print(f"  rrf={h['rrf_score']:.4f}  r_vec={h['r_vec']:>3}  r_txt={h['r_txt']:>3}  "
          f"{h['subject'][:40]:40s}  {h['content'][:100]}")

# Part 4 — Tools & Skills

> 📖 **Guide:** [`docs/part-4-tools-and-skills.md`](docs/part-6-tools-and-skills.md)

> 🔧 **TODO in this part:** **TODO 4** — register `tool_run_sql`

The agent needs to *do things*: execute SQL, run code, write to a scratchpad, fetch a document. In this harness:

- **Tools** live in a vector-indexed `toolbox` table; dispatched by the loop.
- **Skills** live in a vector-indexed `skillbox` table; surfaced as a manifest, body loaded on demand.

Both use the same in-database HNSW index. Vector retrieval keeps the per-turn prompt lean even when the registry grows past 30 tools.

![Toolbox flow](images/cover-toolbox-flow.png)

In [ ]:
# 4.1 — DBFS scratchpad. A real filesystem inside the database (SecureFile LOBs).

import os.path

DBFS_TABLESPACE = "AGENT_DBFS_TS"
DBFS_STORE      = "AGENT_SCRATCH"
DBFS_MOUNT      = "/scratch"

with sys_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM dba_tablespaces WHERE tablespace_name = :n",
                n=DBFS_TABLESPACE)
    if cur.fetchone()[0] == 0:
        cur.execute("SELECT name FROM v$datafile WHERE rownum = 1")
        sample_path = cur.fetchone()[0]
        dbfs_path = f"{os.path.dirname(sample_path)}/agent_dbfs01.dbf"
        cur.execute(f"CREATE TABLESPACE {DBFS_TABLESPACE} DATAFILE '{dbfs_path}' "
                    f"SIZE 100M AUTOEXTEND ON NEXT 50M MAXSIZE 2G")
        print(f"created tablespace {DBFS_TABLESPACE}")
    cur.execute(f"GRANT EXECUTE ON DBMS_DBFS_CONTENT TO {AGENT_USER}")
    cur.execute(f"GRANT EXECUTE ON DBMS_DBFS_SFS TO {AGENT_USER}")
    cur.execute(f"GRANT DBFS_ROLE TO {AGENT_USER}")
    cur.execute(f"ALTER USER {AGENT_USER} QUOTA UNLIMITED ON {DBFS_TABLESPACE}")
sys_conn.commit()

_DBFS_EXISTS_CODES = (955, 64007, 64008, 1)
_dbfs_stmts = [
    ("createfilesystem", f"BEGIN DBMS_DBFS_SFS.CREATEFILESYSTEM("
        f"  store_name => '{DBFS_STORE}', tbl_name => '{DBFS_STORE}_T', "
        f"  tbl_tbs => '{DBFS_TABLESPACE}', use_bf => FALSE); END;"),
    ("registerstore", f"BEGIN DBMS_DBFS_CONTENT.REGISTERSTORE("
        f"  store_name => '{DBFS_STORE}', provider_name => 'sample1', "
        f"  provider_package => 'DBMS_DBFS_SFS'); END;"),
    ("mountstore", f"BEGIN DBMS_DBFS_CONTENT.MOUNTSTORE("
        f"  store_name => '{DBFS_STORE}', store_mount => '{DBFS_MOUNT.lstrip('/')}'); END;"),
]
with agent_conn.cursor() as cur:
    for label, stmt in _dbfs_stmts:
        try:
            cur.execute(stmt)
            print(f"{label}: ok")
        except oracledb.DatabaseError as e:
            err_code = e.args[0].code if e.args else None
            if err_code in _DBFS_EXISTS_CODES or "already exists" in str(e).lower():
                print(f"{label}: (already exists)")
            else:
                print(f"{label}: {e}")
agent_conn.commit()

In [ ]:
class DBFS:
    """Minimal file-like wrapper over Oracle DBFS."""

    def __init__(self, conn, mount: str = DBFS_MOUNT):
        self.conn = conn
        self.mount = mount.rstrip("/")

    def _full(self, path: str) -> str:
        """Resolve to the absolute DBFS path. Accepts both bare and
        mount-qualified forms ('foo.sql' OR '/scratch/foo.sql') so
        callers don't accidentally double the prefix into
        '/scratch/scratch/foo.sql' (ORA-64001).
        """
        if not path.startswith("/"):
            path = "/" + path
        if path == self.mount or path.startswith(self.mount + "/"):
            return path
        return f"{self.mount}{path}"

    def write(self, path: str, content) -> None:
        """Create-or-overwrite the file at `path` under the scratchpad mount."""
        data = content.encode("utf-8") if isinstance(content, str) else content
        full = self._full(path)
        # DELETEFILE(path) is the path-based file delete in DBMS_DBFS_CONTENT;
        # the other arguments (filter, soft_delete, store_name, principal) all
        # default. The exception handler swallows "no such path" so the call
        # is a no-op when the file doesn't exist yet (i.e. "create or overwrite").
        delete_plsql = (
            "BEGIN "
            "  DBMS_DBFS_CONTENT.DELETEFILE(:p); "
            "EXCEPTION WHEN OTHERS THEN NULL; "
            "END;"
        )
        # CREATEFILE(path, properties, content) — properties and content are
        # IN/OUT, so they must be assignment-targets (locals), not literals.
        # We declare them inside the PL/SQL block and seed l_blob from the
        # bound bytes (forced to BLOB via setinputsizes below).
        create_plsql = (
            "DECLARE "
            "  l_props DBMS_DBFS_CONTENT_PROPERTIES_T := DBMS_DBFS_CONTENT_PROPERTIES_T(); "
            "  l_blob  BLOB := :b; "
            "BEGIN "
            "  DBMS_DBFS_CONTENT.CREATEFILE("
            "    path       => :p,"
            "    properties => l_props,"
            "    content    => l_blob); "
            "  COMMIT; "
            "END;"
        )
        with self.conn.cursor() as cur:
            cur.execute(delete_plsql, p=full)
            cur.setinputsizes(b=oracledb.DB_TYPE_BLOB)
            cur.execute(create_plsql, p=full, b=data)
        self.conn.commit()

    def append(self, path: str, content) -> None:
        """Append `content` to `path`. If the file doesn't exist yet,
        behaves like `write`. Use for running findings logs that should
        grow without overwriting prior entries.
        """
        new = content.encode("utf-8") if isinstance(content, str) else content
        try:
            existing = self.read(path).encode("utf-8")
            sep = b"" if existing.endswith(b"\n") or not existing else b"\n"
            self.write(path, existing + sep + new)
        except FileNotFoundError:
            self.write(path, new)

    def read(self, path: str) -> str:
        full = self._full(path)
        # GETPATH(path, properties, content, item_type, ...) - first four are
        # mandatory in this version. We pass locals for all the OUT params and
        # only return content via :out.
        read_plsql = (
            "DECLARE "
            "  l_props     DBMS_DBFS_CONTENT_PROPERTIES_T := DBMS_DBFS_CONTENT_PROPERTIES_T(); "
            "  l_blob      BLOB; "
            "  l_item_type NUMBER; "
            "BEGIN "
            "  DBMS_DBFS_CONTENT.GETPATH("
            "    path       => :p,"
            "    properties => l_props,"
            "    content    => l_blob,"
            "    item_type  => l_item_type); "
            "  :out := l_blob; "
            "END;"
        )
        try:
            with self.conn.cursor() as cur:
                out = cur.var(oracledb.DB_TYPE_BLOB)
                cur.execute(read_plsql, p=full, out=out)
                blob = out.getvalue()
            if blob is None:
                raise FileNotFoundError(full)
            return blob.read().decode("utf-8", errors="replace")
        except oracledb.DatabaseError as e:
            # SFS provider raises ORA-64002 for non-existent paths instead
            # of returning NULL — translate so callers (esp. append) can
            # treat both the same way via FileNotFoundError.
            if e.args and e.args[0].code in (64002, 64007):
                raise FileNotFoundError(full) from e
            raise

    def list(self, path: str = "/") -> list[str]:
        """List every file under `path` in DBFS.

        Two strategies, in order:

        1. ``DBMS_DBFS_CONTENT.LIST(path, '*', 1)`` — the documented API. Works
           on some providers; the SecureFile (SFS) provider on Oracle Free 26ai
           raises ``ORA-64003`` ("unsupported operation") instead.

        2. Direct query on the SFS storage table (``AGENT_SCRATCH_T``).
           Important subtlety: SFS stores pathnames *without* the mount prefix
           (just '/comment.txt', not '/scratch/comment.txt'). We translate the
           caller's mount-qualified `full` path into an SFS-relative prefix
           for the LIKE filter, then prepend the mount back onto each result.
        """
        full = self._full(path)
        out: list[str] = []

        # Strategy 1 — DBMS_DBFS_CONTENT.LIST
        try:
            with self.conn.cursor() as cur:
                cur.execute(
                    "SELECT * FROM TABLE(DBMS_DBFS_CONTENT.LIST(:p, '*', 1))",
                    p=full)
                descs = cur.description or []
                path_idx = next(
                    (i for i, d in enumerate(descs) if "PATH" in (d[0] or "").upper()),
                    0,
                )
                for row in cur:
                    v = row[path_idx]
                    if v: out.append(v)
            if out:
                return out
        except oracledb.DatabaseError:
            pass

        # Strategy 2 — query the SFS storage table directly.
        store_table = f"{DBFS_STORE}_T"
        try:
            with self.conn.cursor() as cur:
                if full == self.mount or full == self.mount + "/":
                    sfs_prefix = "/"
                elif full.startswith(self.mount + "/"):
                    sfs_prefix = full[len(self.mount):]
                    if not sfs_prefix.endswith("/"):
                        sfs_prefix += "/"
                else:
                    sfs_prefix = "/"

                # pathtype = 1 -> file; std_deleted = 0 -> not tombstoned.
                cur.execute(
                    f"SELECT pathname FROM {store_table} "
                    f" WHERE pathtype = 1 AND std_deleted = 0 "
                    f"   AND pathname LIKE :p "
                    f" ORDER BY pathname",
                    p=f"{sfs_prefix}%")
                for (n,) in cur:
                    if n:
                        out.append(f"{self.mount}{n}")
        except oracledb.DatabaseError:
            pass

        return out


# Module-level instance every later cell uses — DBFS is just a thin wrapper
# around `agent_conn`, so it's safe to instantiate once at notebook scope.
scratch = DBFS(agent_conn)
print(f"DBFS scratch ready at {scratch.mount}")


### Pre-built — Oracle MLE sandbox

`exec_js(code)` runs a snippet of JavaScript inside Oracle's Multilingual Engine (MLE). Used for deterministic compute the LLM shouldn't do in its head — percentiles, weighted means, JSON reshaping. No subprocess, no GraalVM on your laptop, same trust boundary as `run_sql`.

In [ ]:
def exec_js(code: str) -> dict:
    """Evaluate a snippet of JavaScript inside Oracle MLE.

    Returns {"stdout": str, "stderr": str, "ok": bool}.
    """
    # Wrap user code so we can capture console.log output and route the
    # result back through mle-js-bindings (the EVAL `result` CLOB doesn't
    # auto-fill from an IIFE on this build, so we use the explicit binding
    # API and read it back via DBMS_MLE.IMPORT_FROM_MLE).
    wrapper = (
        '(function() {\n'
        '  let _stdout = "";\n'
        '  let _stderr = "";\n'
        '  let _ok = true;\n'
        '  const _origLog = console.log;\n'
        '  console.log = function() {\n'
        '    _stdout += Array.from(arguments).map(String).join(" ") + "\\n";\n'
        '  };\n'
        '  try {\n'
        + code + '\n'
        '  } catch (e) {\n'
        '    _stderr = String(e && e.message ? e.message : e) + "\\n" + (e && e.stack ? e.stack : "");\n'
        '    _ok = false;\n'
        '  } finally {\n'
        '    console.log = _origLog;\n'
        '  }\n'
        '  const bindings = require("mle-js-bindings");\n'
        '  bindings.exportValue("result", JSON.stringify({stdout: _stdout, stderr: _stderr, ok: _ok}));\n'
        '})();'
    )

    plsql = (
        "DECLARE "
        "  l_ctx    RAW(16); "
        "  l_result CLOB; "
        "BEGIN "
        "  l_ctx := DBMS_MLE.CREATE_CONTEXT(); "
        "  DBMS_MLE.EVAL(l_ctx, 'JAVASCRIPT', :source); "
        "  DBMS_MLE.IMPORT_FROM_MLE(l_ctx, 'result', l_result); "
        "  :result := l_result; "
        "  DBMS_MLE.DROP_CONTEXT(l_ctx); "
        "EXCEPTION WHEN OTHERS THEN "
        "  BEGIN DBMS_MLE.DROP_CONTEXT(l_ctx); EXCEPTION WHEN OTHERS THEN NULL; END; "
        "  RAISE; "
        "END;"
    )

    with agent_conn.cursor() as cur:
        out = cur.var(oracledb.DB_TYPE_CLOB)
        try:
            cur.setinputsizes(source=oracledb.DB_TYPE_CLOB)
            cur.execute(plsql, source=wrapper, result=out)
        except oracledb.DatabaseError as e:
            return {"stdout": "", "stderr": str(e), "ok": False}

    clob = out.getvalue()
    text = clob.read() if hasattr(clob, "read") else (clob or "")
    if not text:
        return {"stdout": "", "stderr": "MLE returned no output", "ok": False}
    try:
        return json.loads(text)
    except (ValueError, TypeError):
        return {"stdout": text, "stderr": "", "ok": True}


### Pre-built — toolbox DDL + `@register` decorator

The cell below creates the `toolbox` table with an HNSW vector index, defines the `@register` decorator, and provides `retrieve_tools(query, k)` for per-turn tool retrieval.

Read the `@register` body — it's the central abstraction of Part 4.

In [ ]:
import inspect
import re
import typing

TOOLS: dict[str, tuple] = {}            # name -> (callable, openai_schema)
ALWAYS_ON_TOOLS = {"search_knowledge", "run_sql", "remember", "exec_js"}
TOOL_EMBEDDING_DIM = ONNX_EMBED_DIM      # 384 for ALL_MINILM_L12_V2


# ---- toolbox table (DDL) -------------------------------------------------
_TOOLBOX_DDL = [
    (
        "CREATE TABLE toolbox ("
        "  name        VARCHAR2(128) PRIMARY KEY,"
        "  description CLOB NOT NULL,"
        "  parameters  JSON,"
        f" embedding   VECTOR({TOOL_EMBEDDING_DIM}, FLOAT32),"
        "  updated_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP"
        ")"
    ),
    (
        "CREATE VECTOR INDEX toolbox_emb_v ON toolbox(embedding) "
        "ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE"
    ),
]
with agent_conn.cursor() as cur:
    # If a previous run created `toolbox` with a different vector dim,
    # drop it cleanly so the new DDL takes.
    try:
        cur.execute("SELECT data_length FROM user_tab_columns "
                    " WHERE table_name = 'TOOLBOX' AND column_name = 'EMBEDDING'")
        row = cur.fetchone()
        if row and row[0] and row[0] not in (TOOL_EMBEDDING_DIM * 4, None):
            cur.execute("DROP TABLE toolbox PURGE")
            print(f"dropped toolbox (old vector dim) so we can rebuild at {TOOL_EMBEDDING_DIM}")
    except oracledb.DatabaseError:
        pass

    for stmt in _TOOLBOX_DDL:
        try:
            cur.execute(stmt)
        except oracledb.DatabaseError as e:
            code_ = e.args[0].code
            if code_ in (955, 1408):
                continue
            if code_ == 51962 and "VECTOR INDEX" in stmt.upper():
                print("!! ORA-51962: vector_memory_size is 0. HNSW index NOT created.")
                print("   The agent will fall back to linear cosine scan over the toolbox.")
                print("   Fix: run §3.2.1 to set vector_memory_size, `docker restart oracle-free`,")
                print("        restart the kernel, and re-run from §3.1.")
                continue
            raise

agent_conn.commit()


# ---- HNSW verification (loud, explicit) ----------------------------------
# We create the toolbox so the agent can do *vector retrieval* over its own
# tools at runtime - matters once you have 30+ tools and the per-turn prompt
# would otherwise carry every schema. The retrieval is only fast if there's
# an HNSW (Hierarchical Navigable Small World) graph index on `embedding`;
# without it, every turn does a full table scan with VECTOR_DISTANCE.
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT index_name, index_type "
        "  FROM user_indexes "
        " WHERE table_name = 'TOOLBOX' "
        "   AND index_type LIKE 'VECTOR%'"
    )
    vec_indexes = list(cur)

if vec_indexes:
    for name, kind in vec_indexes:
        print(f"OK: HNSW vector index in place -> {name} ({kind})")
else:
    print("!! HNSW vector index is MISSING on toolbox.embedding.")
    print("   Tool retrieval still works (linear cosine over a few rows is fine),")
    print("   but you'll lose the index benefit at scale. To enable: §3.2.1 + bounce + re-run §3.1+.")


# ---- type-hint -> JSON-schema helper -------------------------------------
_PRIMS = {int: "integer", float: "number", bool: "boolean", str: "string"}


def _hint_to_json(hint) -> dict:
    """Map a Python type hint into a JSON-schema fragment (best effort)."""
    origin = typing.get_origin(hint)
    if hint in _PRIMS:
        return {"type": _PRIMS[hint]}
    if origin in (list, typing.List):
        args = typing.get_args(hint) or (str,)
        return {"type": "array", "items": _hint_to_json(args[0])}
    if origin in (dict, typing.Dict):
        return {"type": "object"}
    if origin is typing.Union:
        non_none = [a for a in typing.get_args(hint) if a is not type(None)]
        if len(non_none) == 1:
            return _hint_to_json(non_none[0])
    return {"type": "string"}


def _build_schema(fn) -> tuple[str, str, dict, dict]:
    raw_name = fn.__name__
    name = raw_name[5:] if raw_name.startswith("tool_") else raw_name
    description = (inspect.getdoc(fn) or "").strip()
    if not description:
        raise ValueError(f"tool {name!r} has no docstring; @register needs one for retrieval")
    sig = inspect.signature(fn)
    hints = typing.get_type_hints(fn)
    properties: dict = {}
    required: list[str] = []
    for pname, param in sig.parameters.items():
        prop = _hint_to_json(hints.get(pname, str))
        if param.default is not inspect.Parameter.empty and param.default is not None:
            prop["default"] = param.default
        else:
            required.append(pname)
        properties[pname] = prop
    parameters = {"type": "object", "properties": properties, "required": required}
    openai_schema = {"type": "function",
                     "function": {"name": name,
                                  "description": description,
                                  "parameters": parameters}}
    return name, description, parameters, openai_schema


# ---- @register ------------------------------------------------------------
def register(fn):
    """Argument-less decorator: name from __name__, description from __doc__,
    parameters schema from the signature + type hints. Embedding for the
    `toolbox` row is computed in-DB via VECTOR_EMBEDDING - no Python-side
    embedder, no network call.
    """
    name, description, parameters, openai_schema = _build_schema(fn)
    TOOLS[name] = (fn, openai_schema)

    arg_text = " ".join(parameters["properties"].keys())
    embed_text = f"{name}: {description}\nargs: {arg_text}"

    with agent_conn.cursor() as cur:
        cur.execute(
            "MERGE INTO toolbox t USING (SELECT :tn AS n FROM dual) s ON (t.name = s.n) "
            "WHEN MATCHED THEN UPDATE SET description = :td, parameters = :tp, "
            f"                              embedding = VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA), "
            "                              updated_at = CURRENT_TIMESTAMP "
            "WHEN NOT MATCHED THEN INSERT (name, description, parameters, embedding) "
            f"                       VALUES (:tn, :td, :tp, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA))",
            tn=name, td=description, tp=json.dumps(parameters), etext=embed_text)
    agent_conn.commit()
    return fn


# ---- retrieval ------------------------------------------------------------
def retrieve_tools(query: str, k: int = 6) -> list[dict]:
    """Top-k tool schemas for `query`, plus the always-on set.

    Stage 1: cosine search over `toolbox.embedding` (in-DB; HNSW when present).
    Stage 2: cross-encoder rerank via `rerank()` - in-DB when a reranker is
    loaded (§3.5), pass-through otherwise. Always-on tools are merged in last
    so they're guaranteed in the schema list regardless of rerank ordering.
    """
    # Stage 1: oversample so the reranker has signal to work with.
    cosine_fetch = k * 4
    rows: list[dict] = []
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT name, description FROM toolbox "
            f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
            " FETCH FIRST :k ROWS ONLY",
            q=query, k=cosine_fetch)
        for name, desc in cur:
            desc_text = desc.read() if hasattr(desc, "read") else str(desc or "")
            rows.append({"name": name, "content": desc_text})

    # Stage 2: rerank by description against the user query.
    ranked = rerank(query, rows, top_k=k, content_key="content")

    schemas: dict[str, dict] = {}
    for r in ranked:
        if r["name"] in TOOLS:
            schemas[r["name"]] = TOOLS[r["name"]][1]

    # Always-on set merges in last - guaranteed inclusion.
    for name in ALWAYS_ON_TOOLS:
        if name in TOOLS:
            schemas[name] = TOOLS[name][1]
    return list(schemas.values())


## Register `tool_run_sql`

> 📖 **See:** [Part 4 guide → TODO 4](docs/part-6-tools-and-skills.md#todo-4-register-tool_run_sql)

The agent's primary way to query live data. Must be **read-only** — `SELECT` and `WITH` only. Decorate with `@register`.


In [ ]:
# TODO 4: register tool_run_sql with @register.
# Requirements:
#   - signature: tool_run_sql(sql: str, max_rows: int = 50) -> str
#   - reject any statement that doesn't match _READ_ONLY (already defined)
#   - return JSON: {"columns": [...], "rows": [...], "row_count": N} or {"error": "..."}

_READ_ONLY = re.compile(r"^\s*(select|with)\b", re.IGNORECASE)


@register
def tool_run_sql(sql: str, max_rows: int = 50) -> str:
    """Execute a READ-ONLY SQL statement (SELECT/WITH only) against the target Oracle AI Database
    and return up to `max_rows` rows as JSON. Reject any statement that isn't read-only.
    """
    if not _READ_ONLY.match(sql.strip()):
        return json.dumps({"error": "only SELECT / WITH statements are allowed in run_sql"})
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql)
            cols = [d[0] for d in cur.description]
            rows = []
            for i, r in enumerate(cur):
                if i >= max_rows:
                    break
                rows.append([(v.read() if hasattr(v, "read") else v) for v in r])
        return json.dumps({"columns": cols, "rows": rows, "row_count": len(rows)},
                          default=str)
    except Exception as e:
        return json.dumps({"error": str(e)})

In [ ]:
# ✅ Checkpoint: TODO 4
assert "run_sql" in TOOLS, "❌ TODO 4 incomplete — tool_run_sql not registered"
out = json.loads(tool_run_sql("SELECT 1 AS one FROM dual"))
assert out.get("row_count") == 1, "❌ tool_run_sql didn't execute correctly"
out_bad = json.loads(tool_run_sql("DROP TABLE x"))
assert "error" in out_bad, "❌ tool_run_sql allowed a non-SELECT statement"
print("✅ TODO 4 passed — tool_run_sql registered and read-only enforced")

### Pre-built — the rest of the toolset

`scan_database`, `search_knowledge`, `exec_js`, `scratch_*`, `remember`, `load_skill`, `list_skills`.

In [ ]:
# ============== Tool implementations =====================================

@register
def tool_scan_database(owner: str) -> str:
    """Scan the specified schema of the target Oracle AI Database and update institutional knowledge.
    Run this when the user asks about a schema you have never seen or when you suspect the knowledge store is stale.
    `owner` is the schema owner (e.g. 'DEMO').
    """
    return json.dumps(run_scan(agent_conn, owner=owner))


@register
def tool_search_knowledge(query: str, k: int = 5, kinds: list[str] | None = None) -> str:
    """Search institutional knowledge (what the agent has learned about the target database) by semantic similarity.
    Use this BEFORE running SQL to discover which tables and columns are relevant.
    `kinds` is an optional filter: table, column, relationship, query_pattern, correction.
    """
    hits = retrieve_knowledge(query, k=k, kinds=kinds)
    for h in hits:
        h["body"] = h["body"][:500]
    return json.dumps(hits)


_READ_ONLY = re.compile(r"^\s*(select|with)\b", re.IGNORECASE)


@register
def tool_run_sql(sql: str, max_rows: int = 50) -> str:
    """Execute a READ-ONLY SQL statement (SELECT/WITH only) against the target Oracle AI Database
    and return up to `max_rows` rows as JSON. Reject any statement that isn't read-only.
    """
    if not _READ_ONLY.match(sql.strip()):
        return json.dumps({"error": "only SELECT / WITH statements are allowed in run_sql"})
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql)
            cols = [d[0] for d in cur.description]
            rows = []
            for i, r in enumerate(cur):
                if i >= max_rows: break
                rows.append([(v.read() if hasattr(v, "read") else v) for v in r])
        return json.dumps({"columns": cols, "rows": rows, "row_count": len(rows)},
                          default=str)
    except Exception as e:
        return json.dumps({"error": str(e)})


@register
def tool_exec_js(code: str) -> str:
    """Execute JavaScript inside Oracle MLE (no filesystem, no network).
    Good for arithmetic, string formatting, JSON reshaping, simple aggregations.
    `console.log(...)` output comes back as `stdout`.
    """
    return json.dumps(exec_js(code))


@register
def tool_scratch_write(path: str, content: str) -> str:
    """Write a short-term note to the DBFS scratchpad.
    Use for draft SQL, intermediate results, running calculations you want to reference later in the same turn.
    """
    scratch.write(path, content)
    return json.dumps({"ok": True, "path": path, "bytes": len(content)})


@register
def tool_scratch_append(path: str, content: str) -> str:
    """Append text to the end of a DBFS scratchpad file (or create it).
    Use this instead of `scratch_write` when you want to ADD to a running
    log without losing prior entries — e.g. /scratch/findings.md as you
    discover facts across multiple turns, /scratch/transcript.md, etc.
    For SQL drafts or "latest is truth" content, prefer scratch_write.
    """
    scratch.append(path, content)
    return json.dumps({"ok": True, "path": path, "appended_bytes": len(content)})


@register
def tool_scratch_read(path: str) -> str:
    """Read from the DBFS scratchpad."""
    try:
        return json.dumps({"content": scratch.read(path)})
    except FileNotFoundError:
        return json.dumps({"error": f"not found: {path}"})


@register
def tool_remember(subject: str, body: str, kind: str = "correction") -> str:
    """Persist a correction or learning into institutional knowledge so future turns benefit.
    Use when the user corrects you, or when you discover a non-obvious fact that future retrievals should surface.
    `subject` is a short label (e.g. 'SALES.ORDERS.total_cents'); `body` is the fact written as a full sentence.
    """
    fact = Fact(kind=kind, subject=subject, body=body, metadata={"source": "agent_remember"})
    new, updated, _ = write_facts([fact])
    return json.dumps({"ok": True, "new": new, "updated": updated})


print(f"registered {len(TOOLS)} tools: {sorted(TOOLS)}")


### Pre-built — skillbox table, ingestion, and `load_skill` / `list_skills` tools

The skillbox is seeded from [`oracle/skills`](https://github.com/oracle/skills) — ~155 markdown playbooks across 17 categories. Top-3 names + descriptions are surfaced into every prompt as a manifest; the full body is loaded on demand via `load_skill(name)`.

![Skillbox flow](images/cover-skillbox-flow.png)

In [ ]:
SKILLBOX_DDL = [
    """
    CREATE TABLE skillbox (
      name        VARCHAR2(160) PRIMARY KEY,
      category    VARCHAR2(64),
      description VARCHAR2(2000) NOT NULL,
      body        CLOB NOT NULL,
      source_url  VARCHAR2(2000),
      source_sha  VARCHAR2(64),
      embedding   VECTOR(384, FLOAT32),
      metadata    JSON,
      updated_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """,
    """
    CREATE VECTOR INDEX skillbox_emb_v ON skillbox(embedding)
      ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE
    """,
]

with agent_conn.cursor() as cur:
    for stmt in SKILLBOX_DDL:
        try:
            cur.execute(stmt)
        except oracledb.DatabaseError as e:
            code_ = e.args[0].code
            if code_ in (955, 1408):  # already exists
                continue
            if code_ == 51962 and "VECTOR INDEX" in stmt.upper():
                print("!! ORA-51962: vector_memory_size = 0 — HNSW index on skillbox NOT created.")
                print("   See §3.2.1; skillbox vector retrieval will linear-scan until you bounce.")
                continue
            raise
agent_conn.commit()

# Loud verification (mirrors §10/cell 73's pattern)
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT index_name, index_type FROM user_indexes "
        " WHERE table_name = 'SKILLBOX' AND index_type LIKE 'VECTOR%'"
    )
    vec = list(cur)
if vec:
    for n, t in vec:
        print(f"OK: HNSW vector index in place -> {n} ({t})")
else:
    print("!! HNSW vector index missing on skillbox.embedding.")


In [ ]:
import re as _re_skill

def _parse_skill_md(text: str, fallback_name: str) -> str:
    """Pull a one-paragraph description out of a markdown skill file.

    Looks for the first non-empty paragraph after the H1 (and after any
    horizontal-rule frontmatter delimiters). Falls back to fallback_name when
    nothing useful is in the body.
    """
    lines = text.splitlines()
    n, i = len(lines), 0

    # Skip any YAML frontmatter ("---" ... "---") at the top.
    if i < n and lines[i].strip() == "---":
        i += 1
        while i < n and lines[i].strip() != "---":
            i += 1
        i += 1  # skip closing ---

    # Skip leading blank lines.
    while i < n and not lines[i].strip():
        i += 1
    # Skip the H1 line if present.
    if i < n and lines[i].lstrip().startswith("# "):
        i += 1
    # Skip blank lines after H1.
    while i < n and not lines[i].strip():
        i += 1
    # Collect first paragraph (until a blank line or another heading).
    para = []
    while i < n and lines[i].strip() and not lines[i].lstrip().startswith("#"):
        para.append(lines[i].strip())
        i += 1

    description = " ".join(para).strip()
    if not description:
        description = fallback_name
    if len(description) > 1800:
        description = description[:1797].rsplit(" ", 1)[0] + "..."
    return description


# Smoke test on a synthetic markdown blob.
_test = """# Schema Discovery

Before composing a SQL query against an unfamiliar table, an agent should mine
the catalog views (ALL_TAB_COLUMNS, ALL_CONS_COLUMNS) to confirm column names
and key relationships.

## Steps

1. Query ALL_TABLES.
"""
print(_parse_skill_md(_test, fallback_name="agent/schema-discovery"))


In [ ]:
import urllib.request as _urlreq
import tarfile as _tarfile
import io as _io
import hashlib as _hashlib
import json as _json_skill
import datetime as _dt_skill

ORACLE_SKILLS_REPO = "oracle/skills"
ORACLE_SKILLS_BASE = "db"
SKILLS_TARBALL_URL = f"https://api.github.com/repos/{ORACLE_SKILLS_REPO}/tarball/main"


def _download_repo_tarball(url: str = SKILLS_TARBALL_URL) -> dict[str, bytes]:
    """One unauthenticated request — returns {repo_relative_path: bytes} for every
    .md file under db/. Avoids the 60-req/hr GitHub API limit on per-file fetches.
    """
    req = _urlreq.Request(url, headers={"User-Agent": "eda-skillbox-ingester"})
    with _urlreq.urlopen(req, timeout=60) as resp:
        data = resp.read()

    files: dict[str, bytes] = {}
    with _tarfile.open(fileobj=_io.BytesIO(data), mode="r:gz") as tar:
        for member in tar:
            if not member.isfile():
                continue
            # Tarball paths look like "oracle-skills-<sha7>/db/agent/schema-discovery.md".
            parts = member.name.split("/", 1)
            if len(parts) < 2:
                continue
            rel = parts[1]
            if not rel.startswith(f"{ORACLE_SKILLS_BASE}/") or not rel.endswith(".md"):
                continue
            f = tar.extractfile(member)
            if f is not None:
                files[rel] = f.read()
    return files


def ingest_skills_from_oracle(verbose: bool = True) -> dict:
    """Pull every db/<category>/<skill>.md from oracle/skills and MERGE into skillbox.

    Idempotent: rows with unchanged source_sha are skipped. Run repeatedly without
    side effects.
    """
    if verbose:
        print(f"downloading tarball from {SKILLS_TARBALL_URL} ...")
    raw_files = _download_repo_tarball()
    if verbose:
        print(f"  fetched {len(raw_files)} .md files under db/")

    new = updated = skipped = 0
    skipped_overview = 0

    for rel_path, body_bytes in sorted(raw_files.items()):
        # rel_path = "db/<category>/<file>.md"
        parts = rel_path.split("/")
        if len(parts) != 3:
            # nested deeper than db/<cat>/file.md; skip — the repo isn't deeply nested
            # but be defensive in case future versions add subfolders
            continue
        _, category, filename = parts
        # Skip category-level overview file; only mirror leaf skills
        if filename == "SKILL.md":
            skipped_overview += 1
            continue

        body = body_bytes.decode("utf-8", errors="replace")
        file_stem = filename[:-3]  # drop ".md"
        full_name = f"{category}/{file_stem}"
        sha = _hashlib.sha256(body.encode("utf-8")).hexdigest()[:32]

        # Idempotency: skip if same SHA already in DB
        with agent_conn.cursor() as cur:
            cur.execute(
                "SELECT source_sha FROM skillbox WHERE name = :n",
                n=full_name,
            )
            row = cur.fetchone()
        if row and row[0] == sha:
            skipped += 1
            continue

        description = _parse_skill_md(body, fallback_name=full_name)
        embed_text = f"{full_name}\n{description}"
        source_url = (
            f"https://raw.githubusercontent.com/{ORACLE_SKILLS_REPO}/main/{rel_path}"
        )
        meta = _json_skill.dumps({
            "category": category,
            "ingested_at": _dt_skill.datetime.now(_dt_skill.timezone.utc).isoformat(timespec="seconds"),
            "source_repo": ORACLE_SKILLS_REPO,
            "rel_path": rel_path,
        })

        with agent_conn.cursor() as cur:
            # body is CLOB and skill files routinely exceed VARCHAR2 limits;
            # metadata is JSON. Bind both with explicit types so the thin driver
            # doesn't try to send them as oversized VARCHAR2 payloads (ORA-03146).
            cur.setinputsizes(body=oracledb.DB_TYPE_CLOB, md=oracledb.DB_TYPE_JSON)
            cur.execute(
                "MERGE INTO skillbox t "
                "USING (SELECT :n AS name FROM dual) s ON (t.name = s.name) "
                "WHEN MATCHED THEN UPDATE SET "
                "  category = :cat, "
                "  description = :dsc, "
                "  body = :body, "
                "  source_url = :url, "
                "  source_sha = :sha, "
                f" embedding = VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA), "
                "  metadata = :md, "
                "  updated_at = CURRENT_TIMESTAMP "
                "WHEN NOT MATCHED THEN INSERT "
                "  (name, category, description, body, source_url, source_sha, embedding, metadata) "
                "  VALUES (:n, :cat, :dsc, :body, :url, :sha, "
                f"          VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA), :md)",
                n=full_name, cat=category, dsc=description, body=body,
                url=source_url, sha=sha, etext=embed_text, md=meta,
            )
            if row is None:
                new += 1
                marker = "+"
            else:
                updated += 1
                marker = "~"
        agent_conn.commit()
        if verbose:
            print(f"  {marker} {full_name}")

    return {
        "new": new,
        "updated": updated,
        "skipped": skipped,
        "skipped_overview": skipped_overview,
        "total_in_repo": len(raw_files),
    }

# Run the ingest. Idempotent — re-runs MERGE on changed source_sha and skip
# unchanged files, so this is safe to leave at the bottom of the cell.
_skill_ingest_result = ingest_skills_from_oracle(verbose=False)
print(f"skillbox ingest: {_skill_ingest_result}")


In [ ]:
@register
def tool_load_skill(name: str) -> str:
    """Load the full content of a named skill from the skillbox.
    Use this when the system prompt's "Available skills" manifest lists a skill
    relevant to the current task. The full markdown guide is returned and you
    should follow its instructions for the duration of the task.
    `name` is the full namespace, e.g. "agent/schema-discovery".
    """
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT description, body, source_url, category "
            "  FROM skillbox WHERE name = :n",
            n=name,
        )
        row = cur.fetchone()
    if not row:
        return json.dumps({"error": f"no skill named {name!r}; call list_skills(query) to find available skills"})
    desc, body, url, category = row
    body_text = body.read() if hasattr(body, "read") else str(body or "")
    return json.dumps({
        "name": name,
        "category": category,
        "description": desc,
        "source_url": url,
        "body": body_text,
    })


@register
def tool_list_skills(query: str, k: int = 5) -> str:
    """Search the skillbox semantically. Returns top-k skills (name + description).
    Use when the system prompt's manifest didn't surface the right skill for the
    current task. Then call load_skill(name) on the most relevant one.
    """
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT name, category, description FROM skillbox "
            f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
            " FETCH FIRST :k ROWS ONLY",
            q=query, k=k,
        )
        hits = [{"name": n, "category": c, "description": d} for n, c, d in cur]
    return json.dumps(hits)


# load_skill is always-on so the manifest in the system prompt is never a dead reference.
ALWAYS_ON_TOOLS.add("load_skill")
print(f"registered: load_skill, list_skills  (TOOLS total: {len(TOOLS)}; always-on: {sorted(ALWAYS_ON_TOOLS)})")


In [ ]:
def build_skill_manifest(query: str, k: int = 3) -> str:
    """Top-k skills relevant to `query` formatted as a one-line-per-skill manifest.
    Returns an empty string when the skillbox is empty (graceful degradation —
    the rest of the prompt is unaffected).

    `build_context` (defined in §11) calls this directly, so the agent loop's
    user message starts with the manifest whenever §11.5 has been run.
    """
    try:
        with agent_conn.cursor() as cur:
            cur.execute(
                "SELECT name, description FROM skillbox "
                f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
                " FETCH FIRST :k ROWS ONLY",
                q=query, k=k,
            )
            rows = list(cur)
    except oracledb.DatabaseError:
        return ""
    if not rows:
        return ""
    lines = [f"  - {n} — {(d.read() if hasattr(d, 'read') else d)[:240]}" for n, d in rows]
    return (
        "Available skills (call load_skill(name) to read the full guide and follow it):\n"
        + "\n".join(lines) + "\n\n"
    )


# Part 5 — The Agent Loop

> 📖 **Guide:** [`docs/part-5-agent-loop.md`](docs/part-7-agent-loop.md)

> 🔧 **TODO in this part:** **TODO 5** — `agent_turn`

Everything we've built so far is plumbing. **This** is the agent.

```
build_context → call LLM with retrieved tools → if tool_calls: dispatch  else: final → log
```

Every turn assembles a context block from OAMP + retrieved schema facts, calls the chat LLM with the top-k tools surfaced from the toolbox, and either dispatches the tool calls the model emitted or breaks out with the model's natural-language answer.

In [ ]:
SYSTEM_PROMPT = (
    "You are the Meridian Bank Financial Data Agent — an AML / financial analyst harness operating against an Oracle AI Database.\n"
    "\n"
    "Your job is to answer natural-language questions about the data by reasoning over:\n"
    "  - the institutional knowledge you have accumulated about the database\n"
    "    (tables, columns, relationships, observed query patterns, past corrections,\n"
    "    AND episodic memories of prior conversations on this or other threads);\n"
    "  - the live database, via read-only SQL when you need runtime facts;\n"
    "  - a scratchpad for intermediate notes;\n"
    "  - an in-database JavaScript sandbox via Oracle MLE (`exec_js`) for computation.\n"
    "\n"
    "How to work:\n"
    "  1. ALWAYS call `search_knowledge` first with a paraphrase of the user's question.\n"
    "     Use the results to pick the right tables before writing any SQL.\n"
    "  2. If the user references a schema you have no facts about, call `scan_database`\n"
    "     on that owner to build institutional knowledge, THEN search again.\n"
    "  3. Keep SQL read-only. Use `run_sql` only; never attempt DDL or DML through it.\n"
    "  4. For numeric work over fetched rows - mean / median / percentile / max /\n"
    "     custom aggregation / unit conversion / formatting - you MUST fetch the raw\n"
    "     values with `run_sql` and then call `exec_js` to compute. Do not compute\n"
    "     statistics in your head, and do not lean only on SQL aggregation when JS is\n"
    "     more honest about the math (e.g. percentiles, weighted means, post-fetch\n"
    "     reshaping). The MLE sandbox runs inside Oracle - same trust boundary as\n"
    "     `run_sql`, no network egress, no separate install - so prefer it over\n"
    "     in-head arithmetic every time.\n"
    "  5. For non-trivial SQL (more than ~3 lines, or involving multiple tables /\n"
    "     joins), draft it first to the DBFS scratchpad via `scratch_write` to a\n"
    "     path like `/scratch/<task>.sql`, then `scratch_read` it before passing to\n"
    "     `run_sql`. The scratchpad is a real file system inside Oracle - files\n"
    "     persist across tool calls AND across turns on the same thread.\n"
    "     - `scratch_write(path, content)` REPLACES the file. Use for SQL drafts,\n"
    "       plan revisions, anything where 'latest is the truth'.\n"
    "     - `scratch_append(path, content)` ADDS to the end. Use for running\n"
    "       findings logs (`/scratch/findings.md`), transcripts, anything you\n"
    "       want to grow across multiple steps or turns without overwriting.\n"
    "       BATCH your appends: one `scratch_append` per row of data is wasteful\n"
    "       and burns the iteration budget. Combine many rows into ONE call.\n"
    "  6. When you discover a non-obvious fact, or the user corrects you, call `remember`\n"
    "     so future turns benefit.\n"
    "  7. Prefer short, direct answers. Quote table and column names verbatim.\n"
    "  8. If a tool fails, read the error, adjust, and try once more - don't spin.\n"
    "\n"
    "Never fabricate a table or column. If you're unsure, say so and propose a scan."
)


In [ ]:
import concurrent.futures

# Cache of OAMP thread objects keyed by the harness-level thread_id.
THREADS: dict[str, object] = {}


def get_thread(thread_id: str):
    """Look up or create the OAMP thread for this harness-level thread_id."""
    if thread_id in THREADS:
        return THREADS[thread_id]
    try:
        thread = memory_client.get_thread(thread_id)
    except Exception:
        thread = memory_client.create_thread(
            thread_id=thread_id,
            user_id=USER_ID,
            agent_id=AGENT_ID,
            memory_extraction_frequency=2,
            memory_extraction_window=4,
            enable_context_summary=True,
            context_summary_update_frequency=4,
        )
    THREADS[thread_id] = thread
    return thread


def build_context(thread_id: str, user_query: str, k_knowledge: int = 3) -> str:
    """Assemble the prompt context block for one user turn.

    Three layers stack into one user message:
      1. Skill manifest (top-3 from §11.5's `skillbox`, only if defined yet —
         graceful fallback when the cell hasn't been run).
      2. OAMP context card — relevant memories + recent turns + running summary.
      3. Top-k schema fact memories matching the current question.

    `k_knowledge=3` keeps the per-call prompt small enough that LLM round-trip
    latency stays under the agent_turn budget. Bump it for richer context
    once you've wired prompt caching.
    """
    parts: list[str] = []

    # Layer 1 — skill manifest (defined in §11.5 above). The `try` keeps Part 11
    # runnable even if §11.5 hasn't been executed yet on this kernel.
    try:
        manifest = build_skill_manifest(user_query, k=3)
    except NameError:
        manifest = ""
    if manifest:
        parts.append(manifest.rstrip())

    thread = get_thread(thread_id)

    # Layer 2 — OAMP context card
    card = thread.get_context_card()
    card_text = str(card) if card else ""
    if card_text:
        parts.append("## Memory context (from OAMP)")
        parts.append(card_text)

    # Layer 3 — institutional knowledge top-k
    hits = retrieve_knowledge(user_query, k=k_knowledge)
    if hits:
        parts.append("\n## Institutional knowledge (top matches)")
        for h in hits:
            parts.append(f"- ({h['kind']}) {h['subject']} - {h['body'][:280]}")

    parts.append("\n## User question")
    parts.append(user_query)
    return "\n".join(parts)


# Fire-and-forget logger. OAMP's `add_messages` runs synchronous
# extraction-LLM work when the message-extraction window fills — that's a
# 10–30 s LLM round-trip we don't want to block the agent loop on. We
# submit the write to a small pool and return immediately; any error
# surfaces on the next pool flush via `_drain_log_errors`. Persistence is
# best-effort by design — losing a log line is recoverable; blocking the
# loop on every turn is not.
_LOG_EXECUTOR = concurrent.futures.ThreadPoolExecutor(
    max_workers=4, thread_name_prefix="oamp-log")
_LOG_PENDING: list[concurrent.futures.Future] = []


def _drain_log_errors() -> None:
    """Pop any completed log futures and surface their errors. Cheap;
    runs synchronously but only over already-finished work."""
    still_pending = []
    for fut in _LOG_PENDING:
        if not fut.done():
            still_pending.append(fut)
            continue
        exc = fut.exception()
        if exc is not None:
            print(f"  ! background log_message failed: {type(exc).__name__}: {exc}")
    _LOG_PENDING[:] = still_pending


def log_message(thread_id: str, role: str, content: str) -> None:
    """Persist one chat turn to the OAMP thread without blocking.

    The submit returns immediately; the actual `add_messages` call runs on
    a background thread. We drain completed futures opportunistically so
    failures surface on the next call rather than vanishing.
    """
    from oracleagentmemory.apis.thread import Message

    def _do_log():
        get_thread(thread_id).add_messages([Message(role=role, content=content)])

    _drain_log_errors()
    _LOG_PENDING.append(_LOG_EXECUTOR.submit(_do_log))


## Implement `agent_turn`

> 📖 **See:** [Part 5 guide → TODO 5](docs/part-7-agent-loop.md#todo-5-implement-agent_turn)

The heart of the harness. Read the docs guide once before you start — it walks the dispatch pattern step-by-step. The complete solution is below; type it out, don't copy-paste, so you understand each line.


In [ ]:
# TODO 5: implement agent_turn(user_query, thread_id, max_iterations, budget_seconds, verbose)
# See docs/part-5-agent-loop.md for the full walkthrough.

def agent_turn(user_query: str, thread_id: str = "default",
               max_iterations: int = 8, budget_seconds: float = 360.0,
               verbose: bool = True) -> str:
    started = time.time()
    log_message(thread_id, "user", user_query)

    context = build_context(thread_id, user_query)
    messages: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": context},
    ]
    tool_schemas = retrieve_tools(user_query, k=6)

    final = ""
    step = 0
    for step in range(max_iterations):
        if time.time() - started > budget_seconds:
            if verbose: print(f"  ! budget exhausted at iteration {step}")
            break

        resp = chat(messages, tools=tool_schemas)
        msg = resp.choices[0].message

        if not msg.tool_calls:
            final = msg.content or ""
            if verbose: print(f"  step {step}: final answer")
            break

        # Echo the assistant's tool_calls back so the LLM can match each
        # tool result to the call that produced it.
        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name,
                              "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        # Dispatch each tool the model asked for.
        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            if verbose: print(f"  step {step}: -> {name}({args})")

            if name not in TOOLS:
                output = json.dumps({"error": f"unknown tool: {name}"})
            else:
                fn, _ = TOOLS[name]
                try:
                    output = fn(**args)
                except Exception as e:
                    output = json.dumps({"error": f"{type(e).__name__}: {e}"})

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": output})

    # Forced final-answer path: if budget exhausted, ask one more time without tools.
    if not final:
        messages.append({"role": "user",
                         "content": "Budget exhausted. Provide your best answer now, no more tools."})
        resp = chat(messages, tools=None)
        final = resp.choices[0].message.content or "(no answer produced)"

    log_message(thread_id, "assistant", final)

    # Episodic memory — store the (user, assistant) pair for cross-thread recall.
    try:
        memory_client.add_memory(
            f"User: {user_query}\n\nAssistant: {final}",
            user_id=USER_ID, agent_id=AGENT_ID,
            thread_id=thread_id,
            metadata={"kind": "episodic", "thread_id": thread_id,
                      "user_query": user_query[:240],
                      "elapsed_seconds": round(time.time() - started, 2)},
        )
    except Exception as _e:
        if verbose: print(f"  ! episodic add_memory failed: {type(_e).__name__}: {_e}")

    if verbose: print(f"  [{time.time() - started:.1f}s, {step + 1} steps]")
    return final

## Run the three-turn end-to-end demo

> 📖 **See:** [Part 5 guide → Demo: three-turn conversation](docs/part-7-agent-loop.md#todo-8-run-the-three-turn-demo)

Three turns on the same thread:

1. **Discovery** — forces `search_knowledge` over scanned facts.
2. **Live data** — forces `run_sql`.
3. **Correction + persistence** — forces `remember` and creates a persisted correction memory.


In [ ]:
# Demo: three-turn conversation: run a three-turn conversation on a single thread.

thread = "demo-session-1"

q1 = "What's in the FINANCE schema? Briefly — list the entities and how they relate to each other."
print("USER:", q1)
print("\nASSISTANT:", agent_turn(q1, thread_id=thread))

In [ ]:
q2 = "Which branch regions have the most FLAGGED or BLOCKED transactions? Show me a small table sorted by count desc."
print("USER:", q2)
print("\nASSISTANT:", agent_turn(q2, thread_id=thread))


In [ ]:
q3 = ("Important: in the FINANCE schema, transactions.amount_cents is always USD CENTS — "
      "never dollars. And customers.risk_rating is a 1-100 score, higher means riskier. "
      "Save EACH of these as a separate 'correction' memory by calling the remember tool BEFORE you respond, "
      "then confirm with the memory IDs you got back.")
print("USER:", q3)
print("\nASSISTANT:", agent_turn(q3, thread_id=thread))


# Part 7 — JSON Relational Duality Views

> 📖 **Guide:** [`docs/part-7-duality-views.md`](docs/part-9-duality-views.md)

> 🔧 **TODO in this part:** **TODO 7** — register `tool_get_document`

A **duality view** is a JSON projection over a set of tables joined by PK/FK/UK relationships. The same row in `accounts` is accessible as a **relational tuple** *and* as a **nested JSON document** that includes its `customer`, `branch`, and the arrays of `cards` and `transactions` (with their `merchant` nested inside). One read, no JOINs, no client-side reshaping.

GPT-class models reason about JSON markedly better than tabular join results. And the Part 8 row policy on `transactions.region` is enforced *inside* the duality view by the kernel — same trust boundary, JSON shape on top.

![JSON Relational Duality — three lenses, one source of truth](images/cover-duality-view.png)

### Pre-built — `account_dv` and `customer_dv` DDL

Two read-only duality views on top of `FINANCE`. `account_dv` is the headline document; `customer_dv` is customer-centric (customer + accounts + loans).

In [ ]:
DV_DDL = [
    "DROP VIEW IF EXISTS account_dv",
    "DROP VIEW IF EXISTS customer_dv",

    # account_dv — the headline document. Read-only (no WITH UPDATE clause).
    """
    CREATE OR REPLACE JSON RELATIONAL DUALITY VIEW account_dv AS
    SELECT JSON {
      '_id'          : a.account_id,
      'accountType'  : a.account_type,
      'currency'     : a.currency,
      'balanceCents' : a.balance_cents,
      'status'       : a.status,
      'openedTs'     : a.opened_ts,
      'region'       : (
        SELECT JSON {
          'branchCode' : b.branch_code,
          'name'       : b.name,
          'city'       : b.city,
          'country'    : b.country,
          'region'     : b.region
        } FROM branches b WHERE b.branch_id = a.branch_id
      ),
      'customer'     : (
        SELECT JSON {
          'customerId' : cu.customer_id,
          'fullName'   : cu.full_name,
          'ssn'        : cu.ssn,
          'segment'    : cu.segment,
          'riskRating' : cu.risk_rating
        } FROM customers cu WHERE cu.customer_id = a.customer_id
      ),
      'cards'        : [
        SELECT JSON {
          'cardId'          : ca.card_id,
          'cardType'        : ca.card_type,
          'cardNumber'      : ca.card_number,
          'status'          : ca.status,
          'dailyLimitCents' : ca.daily_limit_cents
        } FROM cards ca WHERE ca.account_id = a.account_id
      ],
      'transactions' : [
        SELECT JSON {
          'txnId'       : t.txn_id,
          'txnTs'       : t.txn_ts,
          'amountCents' : t.amount_cents,
          'currency'    : t.currency,
          'channel'     : t.channel,
          'txnType'     : t.txn_type,
          'status'      : t.status,
          'flagReason'  : t.flag_reason,
          'region'      : t.region,
          'merchant'    : (
            SELECT JSON {
              'merchantId' : m.merchant_id,
              'name'       : m.name,
              'mccCode'    : m.mcc_code,
              'category'   : m.category,
              'country'    : m.country
            } FROM merchants m WHERE m.merchant_id = t.merchant_id
          )
        } FROM transactions t WHERE t.account_id = a.account_id
      ]
    } FROM accounts a
    """,

    # customer_dv — customer-centric. Includes accounts (with branch) and loans.
    """
    CREATE OR REPLACE JSON RELATIONAL DUALITY VIEW customer_dv AS
    SELECT JSON {
      '_id'        : cu.customer_id,
      'fullName'   : cu.full_name,
      'ssn'        : cu.ssn,
      'segment'    : cu.segment,
      'riskRating' : cu.risk_rating,
      'country'    : cu.country,
      'accounts'   : [
        SELECT JSON {
          'accountId'    : a.account_id,
          'accountType'  : a.account_type,
          'currency'     : a.currency,
          'balanceCents' : a.balance_cents,
          'status'       : a.status,
          'branch'       : (
            SELECT JSON {
              'branchCode' : b.branch_code,
              'name'       : b.name,
              'city'       : b.city,
              'region'     : b.region
            } FROM branches b WHERE b.branch_id = a.branch_id
          )
        } FROM accounts a WHERE a.customer_id = cu.customer_id
      ],
      'loans'      : [
        SELECT JSON {
          'loanId'      : l.loan_id,
          'loanType'    : l.loan_type,
          'amountCents' : l.amount_cents,
          'rateBp'      : l.rate_bp,
          'termMonths'  : l.term_months,
          'status'      : l.status
        } FROM loans l WHERE l.customer_id = cu.customer_id
      ]
    } FROM customers cu
    """,
]

# DROP VIEW IF EXISTS isn't valid Oracle syntax — translate manually.
for stmt in DV_DDL:
    if stmt.strip().upper().startswith("DROP VIEW IF EXISTS"):
        view_name = stmt.split()[-1]
        try:
            with demo_conn.cursor() as cur:
                cur.execute(f"DROP VIEW {view_name}")
        except oracledb.DatabaseError:
            pass
        continue
    try:
        with demo_conn.cursor() as cur:
            cur.execute(stmt)
        # Identify which view we just created from the DDL
        head = stmt.strip().split("\n", 1)[0]
        print(f"OK: {head[:80]}")
    except oracledb.DatabaseError as e:
        code_ = e.args[0].code
        head = stmt.strip().split("\n", 1)[0][:80]
        if code_ in (900, 901, 922, 2000):
            print(f"!! {head!r}: duality-view syntax not on this image (ORA-{code_:05d}).")
            print("   The tool_get_document fallback below will return an error pointing")
            print("   the agent at run_sql instead.")
        else:
            raise
demo_conn.commit()

# Verify the views exist
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT view_name FROM all_views "
        " WHERE owner = :o AND view_name IN ('ACCOUNT_DV', 'CUSTOMER_DV')",
        o=DEMO_USER,
    )
    print("duality views in", DEMO_USER, ":", [r[0] for r in cur])


## Register `tool_get_document`

> 📖 **See:** [Part 7 guide → TODO 7](docs/part-9-duality-views.md#todo-7-register-tool_get_document)

Read one full document from a duality view by primary key. The agent calls this instead of writing JOINs whenever it needs the full shape of an entity.

`view` must be one of `account_dv` / `customer_dv`. `key` is the value of `_id`. Return the JSON document as a string, or `{"error": ...}` if the view name is unknown or no document matches.


In [ ]:
# TODO 7: register tool_get_document with @register.
# Whitelist views to {"account_dv", "customer_dv"}. Bind key as int when isdigit().
# Return JSON_SERIALIZE(data PRETTY) from FINANCE.<view>.

@register
def tool_get_document(view: str, key: str) -> str:
    """Read one full document from a JSON Relational Duality View by primary key.
    Use this instead of writing JOINs whenever you need the full shape of an entity
    (an account with its customer/branch/cards/transactions, or a customer with its
    accounts/loans). Returns a JSON document.

    `view` must be one of: account_dv, customer_dv.
    `key` is the value of the document _id (numeric account_id or customer_id, as a string).
    """
    allowed = {"account_dv", "customer_dv"}
    if view not in allowed:
        return json.dumps({"error": f"unknown view {view!r}; allowed: {sorted(allowed)}"})
    try:
        with agent_conn.cursor() as cur:
            cur.execute(
                f"SELECT JSON_SERIALIZE(data PRETTY) FROM {DEMO_USER}.{view} "
                f" WHERE JSON_VALUE(data, '$._id') = :k",
                k=int(key) if str(key).isdigit() else key,
            )
            row = cur.fetchone()
        if not row:
            return json.dumps({"error": f"no document with _id={key} in {view}"})
        body = row[0].read() if hasattr(row[0], "read") else str(row[0])
        return body
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"})

In [ ]:
# ✅ Checkpoint: TODO 7
assert "get_document" in TOOLS, "❌ TODO 7 incomplete — tool_get_document not registered"
out = tool_get_document(view="account_dv", key="1")
import json as _j
assert "_id" in out or "error" in out, "❌ Unexpected response shape"
print("✅ TODO 7 passed — tool_get_document registered")
print(out[:400])

### Pre-built — `tool_query_documents`

Filter a duality view with a SQL predicate. Pre-built so you can move on.

In [ ]:
@register
def tool_query_documents(view: str, where: str = "1=1", max_rows: int = 10) -> str:
    """Filter a JSON Relational Duality View with a SQL predicate.
    Use when you want a list of documents matching some condition without writing
    JOINs by hand. The predicate references underlying-table columns of the view's
    root table (e.g. status, region for account_dv; vessel_type for customer_dv).

    `view` must be one of: account_dv, customer_dv.
    `where` is a SQL boolean expression on the root table's columns (default '1=1').
    `max_rows` caps the result set.
    """
    allowed = {"account_dv", "customer_dv"}
    if view not in allowed:
        return json.dumps({"error": f"unknown view {view!r}; allowed: {sorted(allowed)}"})
    sql = f"SELECT JSON_SERIALIZE(data) FROM {DEMO_USER}.{view} WHERE {where} FETCH FIRST :n ROWS ONLY"
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql, n=max_rows)
            docs = [(r[0].read() if hasattr(r[0], "read") else str(r[0])) for r in cur]
        return json.dumps({"count": len(docs), "documents": [json.loads(d) for d in docs]}, default=str)
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}", "sql": sql})


print(f"registered: query_documents  (TOOLS total: {len(TOOLS)})")

### Demo — same question, the duality-view path

Ask a "give me everything about account 7" question. Watch the trace — the agent should reach for `get_document("account_dv", "7")` and get the full nested document back in one tool call.

In [ ]:
demo_thread_dv = "demo-dv-1"

q_dv = (
    "Give me a complete picture of account_id 7: which customer owns it, which branch "
    "it's at, what cards are attached, and its recent transactions. Use the duality "
    "view if you have one — that's why we built it."
)

print("=" * 70)
print(q_dv)
print("=" * 70)
print(agent_turn(q_dv, thread_id=demo_thread_dv))


### Pre-built — writable view + ETag conflict demo

`voyage_status_dv` adds `WITH UPDATE` to the DV definition, making it writable. Every retrieved document carries `_metadata.etag`; stale writes raise `ORA-42699` automatically. The next two cells demonstrate a clean round-trip and a deliberate two-writer conflict.

In [ ]:
# A SECOND duality view, dedicated to the writable demo.
# Only txn_id (the key) and status are exposed — keeping the surface narrow
# limits what an UPDATE through this view can change. region stays read-only
# so the Part 8 row policy can't be circumvented by mutating it through the
# JSON layer.
DV_WRITABLE_DDL = """
CREATE OR REPLACE JSON RELATIONAL DUALITY VIEW txn_status_dv AS
SELECT JSON {
  '_id'        : t.txn_id,
  'status'     : t.status,
  'region'     : t.region,
  'txnTs'      : t.txn_ts
} FROM transactions t WITH UPDATE

"""

try:
    with demo_conn.cursor() as cur:
        cur.execute(DV_WRITABLE_DDL)
        # AGENT was granted SELECT ANY TABLE in §4 so reads work, but UPDATE
        # through a duality view requires the explicit UPDATE privilege on
        # the view itself (ORA-41900 otherwise). Owner side grants it here.
        cur.execute(f"GRANT SELECT, UPDATE ON txn_status_dv TO {AGENT_USER}")
    demo_conn.commit()
    print(f"OK: created txn_status_dv WITH UPDATE; granted UPDATE to {AGENT_USER}")
except oracledb.DatabaseError as e:
    code_ = e.args[0].code
    if code_ in (900, 901, 922, 2000):
        print(f"!! duality-view DDL not supported on this image (ORA-{code_:05d}). "
              "The §11.6.4 demos below will skip if txn_status_dv isn't present.")
    else:
        raise

# Quick check that it landed
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT view_name FROM all_views WHERE owner = :o AND view_name = 'TXN_STATUS_DV'",
        o=DEMO_USER)
    rows = list(cur)
print(f"txn_status_dv exists: {bool(rows)}")


In [ ]:
import json as _json_dv

# Pick a transaction to round-trip. We capture the initial status so the cell is
# idempotent — we restore it at the end and the rest of the notebook sees no
# side effect.
TARGET_TXN = 1


def _read_doc(txn_id):
    """Fetch one document from txn_status_dv. The result includes
    _metadata.etag, which the kernel sets per-row."""
    with agent_conn.cursor() as cur:
        cur.execute(
            # Aliasing the view as `t` is required so Oracle parses
            # `t.data."_id"` as a JSON path step rather than ambiguous
            # <table>.<column> syntax (which raises ORA-00904).
            f"SELECT JSON_SERIALIZE(t.data PRETTY) "
            f"  FROM {DEMO_USER}.txn_status_dv t "
            f" WHERE JSON_VALUE(t.data, '$_id') = :k",
            k=txn_id,
        )
        row = cur.fetchone()
    if not row:
        return None
    raw = row[0]
    if hasattr(raw, "read"):
        raw = raw.read()
    return _json_dv.loads(raw)


def _put_doc(txn_id, doc):
    """PUT a modified document back through the DV. The kernel compares the
    doc's _metadata.etag against the row's current etag and rejects the UPDATE
    with ORA-42699 if they differ."""
    with agent_conn.cursor() as cur:
        cur.execute(
            f"UPDATE {DEMO_USER}.txn_status_dv t "
            f"   SET t.data = :new_doc "
            f" WHERE JSON_VALUE(t.data, '$_id') = :k",
            new_doc=_json_dv.dumps(doc), k=txn_id,
        )
        rc = cur.rowcount
    agent_conn.commit()
    return rc


# -- Read --
doc = _read_doc(TARGET_TXN)
if doc is None:
    print(f"txn {TARGET_TXN} not found — nothing to demo")
else:
    initial_status = doc["status"]
    initial_etag = doc.get("_metadata", {}).get("etag")
    print(f"Initial state of txn {TARGET_TXN}:")
    print(f"  status: {initial_status}")
    print(f"  etag:   {initial_etag}")

    # -- Modify in memory --
    doc["status"] = "BLOCKED" if initial_status != "BLOCKED" else "COMPLETED"
    new_status = doc["status"]
    print(f"\nFlipping status -> {new_status} and PUTting back with the matching etag...")

    # -- PUT back --
    try:
        rc = _put_doc(TARGET_TXN, doc)
        print(f"  rows updated: {rc}")
    except oracledb.DatabaseError as e:
        print(f"  PUT failed: {e}")
        rc = 0

    # -- Verify --
    after = _read_doc(TARGET_TXN)
    if after:
        print(f"\nAfter PUT — status: {after['status']}, "
              f"new etag: {after.get('_metadata', {}).get('etag')}")
        print(f"  (etag changed: {after.get('_metadata', {}).get('etag') != initial_etag})")

    # -- Restore (idempotency) --
    doc["status"] = initial_status
    _put_doc(TARGET_TXN, doc)
    print(f"\nRestored txn {TARGET_TXN} status to {initial_status}.")


In [ ]:
# Conflict demo: two readers grab the same doc. First writer commits, second
# writer's ETag is now stale and the kernel rejects the UPDATE.
TARGET_TXN = 2

original = _read_doc(TARGET_TXN)
if original is None:
    print(f"txn {TARGET_TXN} not found — nothing to demo")
else:
    initial_status = original["status"]
    print(f"Initial state of txn {TARGET_TXN}: status={initial_status}, "
          f"etag={original.get('_metadata', {}).get('etag')}")

    # Two readers, same etag (because they read at the same logical state).
    reader_a = _read_doc(TARGET_TXN)
    reader_b = _read_doc(TARGET_TXN)
    print(f"\nReader A and Reader B both have etag={reader_a.get('_metadata',{}).get('etag')}")

    # Writer A commits first — succeeds.
    reader_a["status"] = "BLOCKED" if initial_status != "BLOCKED" else "COMPLETED"
    print(f"\nWriter A: PUT status={reader_a['status']} ...")
    try:
        rc_a = _put_doc(TARGET_TXN, reader_a)
        print(f"  rows updated: {rc_a}")
    except oracledb.DatabaseError as e:
        print(f"  PUT failed: {e}")
        rc_a = 0

    # Writer B now tries with its stale etag — must be rejected with ORA-42699.
    reader_b["status"] = "FLAGGED" if initial_status != "FLAGGED" else "COMPLETED"
    print(f"\nWriter B (stale etag): PUT status={reader_b['status']} ...")
    try:
        rc = _put_doc(TARGET_TXN, reader_b)
        print(f"  rows updated: {rc}  (this should NOT have succeeded)")
    except oracledb.DatabaseError as e:
        code_ = e.args[0].code
        if code_ == 42699:
            print(f"  REJECTED with ORA-42699 — stale etag, exactly as expected.")
            print(f"  Writer B would now read again, re-apply its change, and retry.")
        else:
            print(f"  REJECTED with ORA-{code_:05d}: {e}")

    # Restore for cell idempotency
    final = _read_doc(TARGET_TXN)
    final["status"] = initial_status
    _put_doc(TARGET_TXN, final)
    print(f"\nRestored status of txn {TARGET_TXN} to {initial_status}.")


# Part 9 — Tool-Output Offload

> 📖 **Guide:** [`docs/part-9-tool-output-offload.md`](docs/part-11-tool-output-offload.md)

> 🔧 **TODOs in this part (2):** **TODO 8** — `log_tool`; **TODO 9** — register `tool_fetch_tool_output`

Part 5's `agent_turn` inlines every tool result verbatim. Fine for short outputs, **but blows the context window** on a 50-row `run_sql` or a multi-KB skill body. Part 9 fixes that with three pieces:

1. `log_tool` — every dispatch persists the full output as an OAMP memory tagged `kind=tool_output` with the LLM's `tool_call_id`.
2. **Truncation marker** — outputs over 600 bytes are replaced in the message list with a compact preview ending in `...[+N bytes. full output: fetch_tool_output(tool_call_id='call_…')]`.
3. `fetch_tool_output(tool_call_id)` — your TODO 9 — the agent-side retrieval tool that recovers the full bytes by id.

After running this section, `agent_turn` is **redefined** to use the offload + truncation pattern. To revert to the minimal version from Part 5, re-run the Part 5 `agent_turn` cell.

### Pre-built — `log_tool`

Persist the full tool output as an OAMP memory tagged with the `tool_call_id`.

In [ ]:
def log_tool(thread_id: str, tool_call_id: str, tool_name: str,
             tool_args: dict, tool_output: str) -> None:
    """Offload one tool result to OAMP as a kind=tool_output memory."""
    memory_client.add_memory(
        tool_output,
        user_id=USER_ID, agent_id=AGENT_ID,
        thread_id=thread_id,
        metadata={
            "kind": "tool_output",
            "tool_call_id": tool_call_id,
            "tool_name": tool_name,
            "tool_args": json.dumps(tool_args),
        },
    )
print("log_tool ready")

## Register `tool_fetch_tool_output`

> 📖 **See:** [Part 9 guide → TODO 9](docs/part-11-tool-output-offload.md#todo-9-register-tool_fetch_tool_output)

The agent-side retrieval tool. The agent calls it when its inlined preview was truncated and it needs the missing bytes to answer.

Use `memory_client._store.list(..., metadata_filter={"kind": "tool_output", "tool_call_id": tool_call_id}, limit=1)` and return a JSON object with `tool_name`, `tool_args`, `tool_output` if found, or `{"error": ...}` if no record matches.


In [ ]:
# TODO 9: register tool_fetch_tool_output with @register.

@register
def tool_fetch_tool_output(tool_call_id: str) -> str:
    """Retrieve the full, untruncated output of a previous tool call.
    Use this when a prior tool result in your context was truncated with
    '...[+N bytes. full output: fetch_tool_output(tool_call_id=...)]' and you need
    the missing bytes to answer.
    """
    records = memory_client._store.list(
        "memory",
        user_id=USER_ID, agent_id=AGENT_ID,
        metadata_filter={"kind": "tool_output", "tool_call_id": tool_call_id},
        limit=1,
    )
    if not records:
        return json.dumps({"error": f"no tool call with id {tool_call_id}"})
    r = records[0]
    meta = r.metadata or {}
    return json.dumps({
        "tool_name":   meta.get("tool_name"),
        "tool_args":   meta.get("tool_args"),
        "tool_output": r.content,
    })

In [ ]:
# ✅ Checkpoint: TODO 9
assert "fetch_tool_output" in TOOLS, "❌ TODO 9 incomplete — tool_fetch_tool_output not registered"
print("✅ TODO 9 passed — tool_fetch_tool_output registered")

### Pre-built — redefine `agent_turn` with offload + truncation marker

Same loop, but every dispatch goes through `log_tool` (full output → OAMP) and the message list gets a compact preview with a recovery hint when output > 600 bytes.

In [ ]:
# §14.2 — tool-output offload (the missing piece in §11's minimal agent_turn).
#
# Three changes wired together:
#   (a) `log_tool` — persist the full tool output as an OAMP memory tagged
#       kind=tool_output, indexed by the LLM's tool_call_id;
#   (b) `tool_fetch_tool_output` — the agent-side retrieval tool that pulls
#       the bytes back by id when its inlined preview was truncated;
#   (c) redefined `agent_turn` — calls log_tool for each dispatch and replaces
#       large outputs in the message list with a truncation marker that points
#       the agent at fetch_tool_output.
#
# Once you run this cell, all subsequent turns offload + truncate. Re-run
# cell §11's `agent_turn` to revert to the minimal verbatim-inline version.

def log_tool(thread_id: str, tool_call_id: str, tool_name: str,
             tool_args: dict, tool_output: str) -> None:
    """Offload one tool result to OAMP as a kind=tool_output memory."""
    memory_client.add_memory(
        tool_output,
        user_id=USER_ID, agent_id=AGENT_ID,
        thread_id=thread_id,
        metadata={
            "kind": "tool_output",
            "tool_call_id": tool_call_id,
            "tool_name": tool_name,
            "tool_args": json.dumps(tool_args),
        },
    )


@register
def tool_fetch_tool_output(tool_call_id: str) -> str:
    """Retrieve the full, untruncated output of a previous tool call.
    Use this when a prior tool result in your context was truncated with
    '...[+N bytes. full output: fetch_tool_output(tool_call_id=...)]' and you need
    the missing bytes to answer.
    """
    records = memory_client._store.list(
        "memory",
        user_id=USER_ID, agent_id=AGENT_ID,
        metadata_filter={"kind": "tool_output", "tool_call_id": tool_call_id},
        limit=1,
    )
    if not records:
        return json.dumps({"error": f"no tool call with id {tool_call_id}"})
    r = records[0]
    meta = r.metadata or {}
    return json.dumps({
        "tool_name":   meta.get("tool_name"),
        "tool_args":   meta.get("tool_args"),
        "tool_output": r.content,
    })


# Redefine agent_turn to add the offload + truncation marker.
def agent_turn(user_query: str, thread_id: str = "default",     # noqa: F811
                max_iterations: int = 8, budget_seconds: float = 360.0,
                verbose: bool = True) -> str:
    started = time.time()
    log_message(thread_id, "user", user_query)
    context = build_context(thread_id, user_query)
    messages: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": context},
    ]
    tool_schemas = retrieve_tools(user_query, k=6)

    final = ""
    step = 0
    DEDUPE_WINDOW = 3
    recent_calls: list[tuple[str, str]] = []
    for step in range(max_iterations):
        if time.time() - started > budget_seconds:
            if verbose: print(f"  ! budget exhausted at iteration {step}")
            break

        resp = chat(messages, tools=tool_schemas)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            final = msg.content or ""
            if verbose: print(f"  step {step}: final answer")
            break

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name,
                              "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            if verbose: print(f"  step {step}: -> {name}({args})")

            # Dedupe: short-circuit if the LLM is dispatching the same
            # (tool, args) it called within the last DEDUPE_WINDOW steps.
            # Common pathology: agent loops re-reading the same scratchpad
            # path or re-issuing the same search_knowledge query, never
            # converging until budget exhausts.
            call_key = (name, json.dumps(args, sort_keys=True))
            if call_key in recent_calls[-DEDUPE_WINDOW:]:
                output = json.dumps({
                    "note": f"duplicate call to {name} with identical args within "
                            f"the last {DEDUPE_WINDOW} dispatches — output unchanged; "
                            f"reuse the prior tool result above instead of re-dispatching."
                })
                if verbose: print(f"          ↳ duplicate dispatch — short-circuited")
            elif name not in TOOLS:
                output = json.dumps({"error": f"unknown tool: {name}"})
            else:
                fn, _ = TOOLS[name]
                try:
                    output = fn(**args)
                except Exception as e:
                    output = json.dumps({"error": f"{type(e).__name__}: {e}"})
            recent_calls.append(call_key)

            # Offload full bytes, inline a compact preview with a recovery hint.
            log_tool(thread_id, tc.id, name, args, output)
            if len(output) <= 600:
                preview = output
            else:
                preview = (
                    output[:600] +
                    f" ...[+{len(output)-600} bytes. "
                    f"full output: fetch_tool_output(tool_call_id='{tc.id}')]"
                )
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": preview})

    if not final:
        messages.append({"role": "user",
                         "content": "Budget exhausted. Provide your best answer now, no more tools."})
        resp = chat(messages, tools=None)
        final = resp.choices[0].message.content or "(no answer produced)"

    log_message(thread_id, "assistant", final)

    try:
        memory_client.add_memory(
            f"User: {user_query}\n\nAssistant: {final}",
            user_id=USER_ID, agent_id=AGENT_ID,
            thread_id=thread_id,
            metadata={"kind": "episodic", "thread_id": thread_id,
                      "user_query": user_query[:240],
                      "elapsed_seconds": round(time.time() - started, 2)},
        )
    except Exception as _e:
        if verbose: print(f"  ! episodic add_memory failed: {type(_e).__name__}: {_e}")

    if verbose: print(f"  [{time.time() - started:.1f}s, {step + 1} steps]")
    return final


# Sanity: ask for the full output of the most recent tool call on `thread`.
recent = memory_client._store.list(
    "memory",
    user_id=USER_ID, agent_id=AGENT_ID,
    thread_id=thread,
    metadata_filter={"kind": "tool_output"},
    limit=1,
)
if recent:
    last_id = (recent[0].metadata or {}).get("tool_call_id", "")
    print(f"fetching tool_call_id={last_id}")
    print(tool_fetch_tool_output(tool_call_id=last_id)[:400])
else:
    print("(no tool_output memories yet on this thread — run a turn first)")


# Closing thoughts — and the running app

You've built and exercised every component of the harness:

- Long-term memory survives between turns and across sessions, because OAMP persists in Oracle.
- Retrieval combines semantic similarity (vector) with exact-token precision (Oracle Text), fused via RRF in one SQL statement.
- Tools are vector-indexed; the registry can grow past 30 tools without bloating the per-turn prompt.
- The agent dispatches `run_sql`, `exec_js`, `scratch_*`, and `remember` through a 90-line loop.
- JSON Relational Duality Views serve the same row as a tuple AND a nested document.
- Tool-output offload + truncation markers keep the context window stable on long, data-heavy turns.

## See it running — open the app

The Codespace started a Flask + React deployment of this same harness on first launch. Open it now:

> **http://localhost:3000** (auto-forwarded — check the **PORTS** tab at the bottom of VS Code if it didn't open in a preview)

The app uses the same Oracle, the same OAMP store, the same `toolbox` and `skillbox` you populated in this notebook. What's different:

- A real chat UI, with each tool dispatch shown as a separate bubble.
- A live **memory pane** on the right — top semantic memories, recent tool outputs, skill manifest, token usage — refreshed after every turn.
- An **identity selector** in the header — pick a persona (`cfo`, `analyst.east`, `analyst.west`, `ops.viewer`) and the same SQL returns different rows.
- A **3D globe** the agent can drive via the `focus_world` tool.
- Live web/news access via the `search_tavily` tool (set `TAVILY_API_KEY` to enable).

### Try these prompts

| Prompt | Exercises |
|---|---|
| *"What's in the FINANCE schema?"* | scanner-built institutional knowledge (Part 2) |
| *"Which branch regions have the most FLAGGED transactions?"* | `run_sql` (Part 4 + Part 5) |
| *"Pull every FLAGGED transaction's amount and use exec_js to compute mean / median in dollars."* | `run_sql` → `exec_js` (Oracle MLE) |
| *"Give me the complete document for account 7 — use the duality view."* | `get_document("account_dv", "7")` (Part 7) |
| *"How do I diagnose ORA-00904? Consult any guide you have."* | `load_skill` from the skillbox |

For the full app architecture, read [`app/README.md`](app/README.md).

## What's left if you want to go further

- **Replace the regex SQL guard with a real parser.** `antlr` with Oracle's grammar, or a stage that runs `EXPLAIN PLAN FOR` and mines `plan_table` for accessed objects.
- **Layer evaluations on top.** Pair each golden question with a target SQL, execute both, grade with the LLM. Regressions surface as failed grades in CI.
- **Tune OAMP's extraction cadence.** `memory_extraction_frequency` and `context_summary_update_frequency` trade fidelity against LLM cost.
- **Push policy further into the kernel.** The duality views compose: try adding writable duality views with end-user-aware constraints (only `cfo` can mark voyages COMPLETED, etc).

The harness is small on purpose. Make it boring, make it legible, and let the model do the interesting work.